# Mathematics for Machine Learning
### From first-principles math to the math behind modern AI

34 runnable, **seeded** programs that build the mathematics of machine learning from the ground up: **Foundations** (numbers, functions, geometry, logic, limits), **Linear Algebra** (vectors, matrices, eigen/SVD/PCA, tensors), **Calculus & Optimization** (derivatives, gradients, the chain rule, autodiff, backprop, gradient descent, convexity), **Probability & Statistics** (distributions, Bayes, sampling, hypothesis testing, information theory), the **Math of ML** (linear/logistic regression, SVMs/trees, neural nets/CNNs, embeddings), and **Modern AI** (Fourier/graphs/Markov, attention/transformers, diffusion, reinforcement learning). Every number on screen comes from a real run — these notebooks reproduce them to the digit.

Run the setup cell once, then any episode top to bottom.

In [ ]:
!pip install -q numpy scipy matplotlib torch

## Setup · write every episode program to disk

In [ ]:
%%writefile ep01_neuron.py
"""
M01 · Numbers & Algebra for ML — a linear neuron is a weighted sum (Sigma).
A weight is a number, a prediction is a sum, a loss is one number measuring wrongness.
We compute the same weighted sum two ways (a hand loop and NumPy's dot), add a squared-error
loss, and draw each term. Seeded, offline. Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m01.png"

rng = np.random.default_rng(0)                          # b1: reproducible
x = np.array([2.0, -1.0, 3.0])                          #     three inputs
w = np.array([0.5, 1.5, -0.5])                          #     three weights

s_loop = 0.0                                            # b2: Sigma by hand
for wi, xi in zip(w, x):
    s_loop += wi * xi
s_dot = float(w @ x)                                    # b3: same via dot product

target = 1.0                                            # b4: squared-error loss
loss = (s_dot - target) ** 2
log_loss = np.log(loss)
terms = w * x                                           # b5: each wi*xi

print(f"loop sum   = {s_loop:.4f}")
print(f"dot sum    = {s_dot:.4f}")
print(f"terms      = {np.round(terms, 4)}")
print(f"loss (MSE) = {loss:.4f}")
print(f"log loss   = {log_loss:.4f}")
assert abs(s_loop - s_dot) < 1e-9, "loop and dot must agree"

# ---- hero figure: each weighted term + the total (figure == terminal) ----
AMBER, INK, DIM = "#FFB454", "#1b2330", "#7d8693"
fig, ax = plt.subplots(figsize=(6.6, 4.2), dpi=130)
bars = ax.bar([f"w{i+1}·x{i+1}" for i in range(3)], terms, color=AMBER, edgecolor="#b97f2e", lw=1, width=0.6)
for b, v in zip(bars, terms):
    ax.text(b.get_x() + b.get_width() / 2, v + (0.06 if v >= 0 else -0.06),
            f"{v:.1f}", ha="center", va="bottom" if v >= 0 else "top",
            fontsize=12, fontweight="bold", color=INK)
ax.axhline(0, color="#888", lw=0.9)
ax.axhline(s_dot, color=AMBER, ls="--", lw=1.6)
ax.text(2.4, s_dot, f"sum = {s_dot:.1f}", ha="right", va="center", fontsize=12,
        fontweight="bold", color="#b97f2e")
ax.set_ylim(-2.2, 1.6)
ax.set_ylabel("weighted term  wᵢ·xᵢ")
ax.set_title(f"a neuron = weighted sum = {s_dot:.1f}   (MSE loss {loss:.1f})",
             fontsize=13, fontweight="bold", color=INK)
for sp in ["top", "right"]:
    ax.spines[sp].set_visible(False)
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white")
print(f"wrote {FIG.name}  (sum={s_dot:.1f}, loss={loss:.1f})")


In [ ]:
%%writefile ep02_functions.py
"""
M02 · Functions & Graphs — a function is a rule that turns an input into one output.
We plot three functions (sigmoid, ReLU, and the composition sigmoid(linear)) on one
axis, evaluate each at a probe point, and confirm a composition computed two ways agrees.
Pure math (no randomness), offline. Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m02.png"

x = np.linspace(-6, 6, 200)                            # b1: shared domain
linear  = lambda t: 2.0 * t - 1.0                      # b2: three functions
sigmoid = lambda t: 1.0 / (1.0 + np.exp(-t))
relu    = lambda t: np.maximum(0.0, t)

p = 2.0                                                # b3: probe at x = 2
print(f"linear(2)  = {linear(p):.4f}")
print(f"sigmoid(2) = {sigmoid(p):.4f}")
print(f"relu(2)    = {relu(p):.4f}")

comp_nested = sigmoid(linear(p))                       # b4: f(g(x)) two ways
comp_byhand = 1.0 / (1.0 + np.exp(-(2.0 * p - 1.0)))
print(f"sigmoid(linear(2)) nested = {comp_nested:.4f}")
print(f"sigmoid(linear(2)) byhand = {comp_byhand:.4f}")
assert abs(comp_nested - comp_byhand) < 1e-12, "two ways must agree"

# ---- hero figure: three curves + the probe point (figure == terminal) ----
AMB, AMB2, AMB3, INK, DIM = "#FFB454", "#FFCB80", "#C8821E", "#1b2330", "#8a8fa3"
fig, ax = plt.subplots(figsize=(6.4, 4.2), dpi=130)
ax.plot(x, sigmoid(x),         color=AMB,  lw=2.6, label="sigmoid(x)")
ax.plot(x, relu(x),            color=AMB2, lw=2.6, label="ReLU(x)")
ax.plot(x, sigmoid(linear(x)), color=AMB3, lw=2.6, label="sigmoid(linear(x))")
ax.scatter([p], [comp_nested], color=AMB, edgecolor=INK, s=90, zorder=6)
ax.annotate(f"(2, {comp_nested:.3f})", (p, comp_nested), textcoords="offset points",
            xytext=(10, -16), fontsize=11, fontweight="bold", color=INK)
ax.axhline(0, color="#bbb", lw=0.8); ax.axhline(1, color="#ddd", lw=0.8, ls="--")
ax.set_xlabel("input  x", color=INK); ax.set_ylabel("output  f(x)", color=INK)
ax.set_title(f"a model is a function you can draw  ·  sigmoid(linear(2)) = {comp_nested:.3f}",
             color=INK, fontsize=12, fontweight="bold")
ax.legend(loc="center left", framealpha=0.9, fontsize=10)
ax.set_ylim(-1.2, 3.2); ax.grid(alpha=0.12)
for sp in ["top", "right"]:
    ax.spines[sp].set_visible(False)
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white", bbox_inches="tight")
print(f"saved {FIG.name}  (sigmoid(linear(2)) = {comp_nested:.3f})")


In [ ]:
%%writefile ep03_geometry.py
"""
M03 · Trigonometry & Geometry for ML — distance and angle, the two ways a model compares.
Three fixed 2-D vectors: Euclidean distance, then cosine similarity for an aligned pair and a
perpendicular pair, drawn as arrows so the angle is visible. Pure math (no randomness), offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m03.png"

a = np.array([3.0, 1.0])                               # b1: three fixed vectors
b = np.array([2.0, 1.0])                               #     nearly aligned with a
c = np.array([-1.0, 3.0])                              #     ~perpendicular to a

dist_ac = np.linalg.norm(a - c)                        # b2: Euclidean distance

def cosine(u, v):                                      # b3: cosine similarity
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

cos_ab = cosine(a, b)                                  # b4: two pairs + angles
cos_ac = cosine(a, c)
ang_ab = np.degrees(np.arccos(cos_ab))
ang_ac = np.degrees(np.arccos(cos_ac))
print(f"distance(a,c) = {dist_ac:.4f}")
print(f"cos(a,b) = {cos_ab:.4f}  angle = {ang_ab:.2f} deg")
print(f"cos(a,c) = {cos_ac:.4f}  angle = {ang_ac:.2f} deg")

# ---- hero figure: three arrows from the origin (figure == terminal) ----
INK, DIM = "#1b2330", "#8a8fa3"
fig, ax = plt.subplots(figsize=(5.6, 5.6), dpi=130)
for v, col, name in [(a, "#FFB454", "a"), (b, "#FFCB80", "b"), (c, "#C8821E", "c")]:
    ax.annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=col, lw=2.8))
    ax.text(v[0] * 1.06, v[1] * 1.06, name, color=col, fontsize=16, fontweight="bold")
ax.text(0.5, 1.0, f"a·b angle {ang_ab:.0f}°", color="#C8821E", fontsize=11, rotation=18)
ax.text(-0.95, 1.7, f"a·c angle {ang_ac:.0f}°", color="#C8821E", fontsize=11)
ax.set_title(f"distance(a,c)={dist_ac:.2f}   ·   cos(a,b)={cos_ab:.2f}   cos(a,c)={cos_ac:.2f}",
             color=INK, fontsize=12, fontweight="bold")
ax.axhline(0, color="#bbb", lw=0.8); ax.axvline(0, color="#bbb", lw=0.8)
ax.set_xlim(-2, 4); ax.set_ylim(-1, 4); ax.set_aspect("equal")
ax.grid(alpha=0.15); ax.set_xlabel("x", color=INK); ax.set_ylabel("y", color=INK)
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white", bbox_inches="tight")
print(f"saved {FIG.name}  (cos(a,b)={cos_ab:.2f}, cos(a,c)={cos_ac:.2f})")


In [ ]:
%%writefile ep04_sets.py
"""
M04 · Set Theory & Logic — precision and recall are just set overlap.
Two sets (actually-positive, predicted-positive); their intersection is the true positives.
precision = TP / predicted, recall = TP / actual. Seeded, offline. Draws the Venn.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m04.png"

rng = np.random.default_rng(4)                                 # b1: reproducible
N = 20
actual = set(np.flatnonzero(rng.random(N) < 0.5).tolist())    #     actually positive
predicted = set(np.flatnonzero(rng.random(N) < 0.5).tolist()) #     predicted positive

tp = actual & predicted                                       # b2: intersection
precision = len(tp) / len(predicted)                          # b3
recall = len(tp) / len(actual)

print(f"actual={len(actual)} predicted={len(predicted)} TP={len(tp)}")   # b4
print(f"precision={precision:.3f}")
print(f"recall={recall:.3f}")

# ---- hero figure: two-circle Venn (figure == terminal) ----
AMB, INK, DIM = "#FFB454", "#1b2330", "#8a8fa3"
fig, ax = plt.subplots(figsize=(6.2, 4.4), dpi=130)
ax.add_patch(Circle((0.40, 0.5), 0.30, fc=AMB, alpha=0.30, ec=AMB, lw=2.2))
ax.add_patch(Circle((0.60, 0.5), 0.30, fc=AMB, alpha=0.30, ec=AMB, lw=2.2))
ax.text(0.17, 0.5, f"actual\n{len(actual)}", ha="center", va="center", fontsize=13, color=INK, fontweight="bold")
ax.text(0.83, 0.5, f"predicted\n{len(predicted)}", ha="center", va="center", fontsize=13, color=INK, fontweight="bold")
ax.text(0.5, 0.5, f"TP\n{len(tp)}", ha="center", va="center", fontsize=15, color="#9a5b12", fontweight="bold")
ax.text(0.5, 0.92, "true positives = actual ∩ predicted", ha="center", va="center", fontsize=11, color=DIM)
ax.text(0.5, 0.06, f"precision = TP/predicted = {precision:.3f}    recall = TP/actual = {recall:.3f}",
        ha="center", va="center", fontsize=11, color=INK)
ax.set_title(f"precision = {precision:.3f}   ·   recall = {recall:.3f}", color=INK, fontsize=13, fontweight="bold")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal"); ax.axis("off")
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white", bbox_inches="tight")
print(f"saved {FIG.name}  (precision={precision:.3f}, recall={recall:.3f})")


In [ ]:
%%writefile ep05_series.py
"""
M05 · Sequences, Series & Limits — a geometric series converges to a/(1-r).
The terms 1, 1/2, 1/4, ... are a sequence; their running totals are a series; the value
those totals approach is the limit. We sum term-by-term and compare to the closed form.
Pure math (no randomness), offline. Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m05.png"

a, r = 0.5, 0.5                              # b1: first pour = half a glass, ratio |r| < 1
n = np.arange(12)
terms = a * r ** n                           # the sequence of terms
partial = np.cumsum(terms)                   # b2: running sums (the series)

limit = a / (1 - r)                          # b3: closed-form limit
gap = limit - partial[-1]                    # b4: how far the last sum still is

print(f"limit a/(1-r) = {limit:.6f}")
print(f"partial_sum[11] = {partial[-1]:.6f}")
print(f"gap = {gap:.6f}")

# ---- hero figure: partial sums crowding the limit line (figure == terminal) ----
AMB, INK, DIM = "#FFB454", "#1b2330", "#8a8fa3"
fig, ax = plt.subplots(figsize=(6.4, 4.2), dpi=130)
ax.axhline(limit, ls="--", color="#999", lw=1.6, label=f"limit = {limit:.3f}")
ax.plot(n, partial, "o-", color=AMB, lw=2.4, ms=8, label="partial sums")
ax.annotate(f"{partial[-1]:.4f}", (n[-1], partial[-1]), textcoords="offset points",
            xytext=(-14, -20), fontsize=11, fontweight="bold", color=INK)
ax.text(5.5, limit + 0.04, f"gap = {gap:.6f}  (never quite reaches)", fontsize=10, color=DIM)
ax.set_xlabel("term index  n", color=INK); ax.set_ylabel("partial sum", color=INK)
ax.set_title(f"geometric series  ->  {limit:.3f}", color=INK, fontsize=13, fontweight="bold")
ax.set_ylim(0.3, 1.18); ax.grid(alpha=0.13); ax.legend(loc="lower right", fontsize=10)
for sp in ["top", "right"]:
    ax.spines[sp].set_visible(False)
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white", bbox_inches="tight")
print(f"saved {FIG.name}  (limit={limit:.3f}, final={partial[-1]:.4f})")


In [ ]:
%%writefile ep06_vectors.py
"""
M06 · Vectors & Vector Spaces — a vector is a list of numbers AND an arrow.
Two operations define everything: add (tip-to-tail) and scale. We add and scale two 2-D
vectors, test independence via the determinant of [u v], and draw the parallelogram.
Pure math (no randomness), offline. Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m06.png"

u = np.array([3.0, 2.0])                     # b1
v = np.array([1.0, 4.0])

s = u + v                                    # b2: tip-to-tail sum
scaled = 2 * u                               # scaling

M = np.column_stack([u, v])                  # b3: independence test
det = np.linalg.det(M)
independent = abs(det) > 1e-9

print(f"u+v = {s}")                          # b4
print(f"2u  = {scaled}")
print(f"det = {det:.1f}  independent = {independent}")

# ---- hero figure: u, v, u+v arrows + the tip-to-tail parallelogram ----
BLUE, PURP, INK = "#7cc1ff", "#bd7dff", "#1b2330"
fig, ax = plt.subplots(figsize=(5.4, 5.4), dpi=130)
ax.plot([u[0], s[0]], [u[1], s[1]], color=BLUE, lw=1.4, ls="--", alpha=0.6)   # v shifted to u tip
ax.plot([v[0], s[0]], [v[1], s[1]], color=BLUE, lw=1.4, ls="--", alpha=0.6)   # u shifted to v tip
for vec, c, lab in [(u, BLUE, "u=(3,2)"), (v, BLUE, "v=(1,4)"), (s, PURP, "u+v=(4,6)")]:
    ax.annotate("", xy=vec, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=c, lw=2.8))
    ax.text(vec[0] + 0.12, vec[1] + 0.12, lab, color=c, fontsize=12, fontweight="bold")
ax.axhline(0, color="#bbb", lw=0.8); ax.axvline(0, color="#bbb", lw=0.8)
ax.set_xlim(-1, 6); ax.set_ylim(-1, 7); ax.set_aspect("equal"); ax.grid(alpha=0.18)
ax.set_xlabel("x", color=INK); ax.set_ylabel("y", color=INK)
ax.set_title(f"u+v = {s}    det = {det:.1f}  (independent = {independent})",
             color=INK, fontsize=12, fontweight="bold")
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white", bbox_inches="tight")
print(f"saved {FIG.name}  (u+v={s}, det={det:.1f})")


In [ ]:
%%writefile ep07_matrices.py
"""
M07 · Matrices & Matrix Operations — a linear layer is one matmul, and the inverse undoes it.
We push a batch X (4 examples, 3 features) through Y = X @ W + b in a single matmul, then
invert W to recover the pre-bias signal exactly. Seeded, offline. Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m07.png"

rng = np.random.default_rng(7)
X = rng.integers(0, 5, size=(4, 3)).astype(float)          # b1: batch — 4 examples, 3 features
W = np.array([[2., 0., 1.], [0., 3., 0.], [1., 0., 2.]])   # b2: weights (3x3)
b = np.array([1., -1., 0.])                                #     bias (1x3)
Y = X @ W + b                                              # b3: forward pass, one matmul

Winv = np.linalg.inv(W)                                    # b4: invert and recover
X_rec = (Y - b) @ Winv
err = float(np.max(np.abs(X - X_rec)))
rank = int(np.linalg.matrix_rank(W))

print("X shape:", X.shape, " W shape:", W.shape, " Y shape:", Y.shape)   # b5
print("rank(W):", rank)
print("max reconstruction error:", round(err, 12))

# ---- hero figure: Y heatmap (figure == terminal) ----
fig, ax = plt.subplots(figsize=(5.4, 4.2), dpi=130)
im = ax.imshow(Y, cmap="Blues")
ax.set_title("Y = X @ W + b   (batch through a linear layer)", fontsize=12, fontweight="bold", color="#1b2330")
ax.set_xlabel("output feature"); ax.set_ylabel("example")
ax.set_xticks(range(3)); ax.set_yticks(range(4))
for (i, j), v in np.ndenumerate(Y):
    ax.text(j, i, f"{v:.0f}", ha="center", va="center",
            color="#07090f" if v < Y.max() * 0.6 else "white", fontsize=13, fontweight="bold")
fig.colorbar(im, ax=ax, label="value")
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, dpi=130, facecolor="white")
print(f"saved {FIG.name}  (Y shape {Y.shape}, rank {rank}, err {err})")


In [ ]:
%%writefile ep08_transforms.py
"""
M08 · Linear Transformations — a matrix is a verb that warps space.
We rotate (45 deg) and non-uniformly scale the unit square, compose the two as one matrix,
and confirm det(M) = det(R) * det(S) — the determinant is the area-scaling factor.
Pure math (no randomness), offline. Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m08.png"

t = np.pi / 4                                              # b1: 45-deg rotation + non-uniform scale
R = np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
S = np.array([[2.0, 0.0], [0.0, 1.5]])
sq = np.array([[0, 1, 1, 0, 0], [0, 0, 1, 1, 0]], dtype=float)   # b2: unit-square corners (closed)
M = R @ S                                                 # b3: compose — apply S first, then R
out = M @ sq

dR, dS, dM = np.linalg.det(R), np.linalg.det(S), np.linalg.det(M)   # b4
print("det(R):", round(dR, 6))
print("det(S):", round(dS, 6))
print("det(M):", round(dM, 6))
print("det(R)*det(S):", round(dR * dS, 6))
print("match:", bool(np.isclose(dM, dR * dS)))

# ---- hero figure: original unit square vs transformed parallelogram ----
BLUE, GREY, INK = "#7cc1ff", "#7d8693", "#1b2330"
fig, ax = plt.subplots(figsize=(5.4, 5.4), dpi=130)
ax.fill(sq[0], sq[1], color=GREY, alpha=0.25)
ax.plot(sq[0], sq[1], color=GREY, lw=2, label="unit square (area 1)")
ax.fill(out[0], out[1], color=BLUE, alpha=0.30)
ax.plot(out[0], out[1], color=BLUE, lw=2.6, label="M @ square")
ax.set_title(f"linear transform: area x{dM:.2f}   (det R={dR:.0f} · det S={dS:.0f})",
             color=INK, fontsize=12, fontweight="bold")
ax.set_aspect("equal"); ax.axhline(0, color="#888", lw=0.8); ax.axvline(0, color="#888", lw=0.8)
ax.set_xlim(-2.5, 2.5); ax.set_ylim(-1, 3.2); ax.grid(alpha=0.15)
ax.legend(loc="lower left", fontsize=10)
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white", bbox_inches="tight")
print(f"saved {FIG.name}  (det(M)={dM:.2f} = {dR:.0f} x {dS:.0f})")


In [ ]:
%%writefile ep09_norms.py
"""
M09 · Norms, Dot Products & Projections — the three rulers of vector space.
On a concrete pair A=(3,4), B=(4,0): L1/L2 norms, the dot product, the cosine of the angle,
and the projection of A onto B (the "shadow" A casts on B). Pure math, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m09.png"

A = np.array([3.0, 4.0])                     # b1: two fixed vectors
B = np.array([4.0, 0.0])

l1 = float(np.sum(np.abs(A)))                # b2: L1 and L2 norms of A
l2 = float(np.sqrt(np.sum(A ** 2)))

dot = float(A @ B)                           # b3: dot product and cosine
cos = dot / (np.linalg.norm(A) * np.linalg.norm(B))

proj = (A @ B) / (B @ B) * B                 # b4: projection of A onto B
proj_len = float(np.linalg.norm(proj))

print("L1(A):", round(l1, 4))
print("L2(A):", round(l2, 4))
print("A . B:", round(dot, 4))
print("cosine:", round(cos, 4))
print("proj of A onto B:", np.round(proj, 4))
print("|proj|:", round(proj_len, 4))

# ---- hero figure: A, B, the projection, and the drop-line ----
BLUE, PURP, GOLD, GREY, INK = "#7cc1ff", "#bd7dff", "#ffd166", "#7d8693", "#1b2330"
fig, ax = plt.subplots(figsize=(5.4, 5.4), dpi=130)
ax.annotate("", xy=A, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=BLUE, lw=2.8))
ax.annotate("", xy=B, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=PURP, lw=2.8))
ax.annotate("", xy=proj, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=GOLD, lw=3.4))
ax.plot([A[0], proj[0]], [A[1], proj[1]], "--", color=GREY, lw=1.6)
ax.text(A[0] + 0.1, A[1] + 0.1, "A=(3,4)", color=BLUE, fontsize=12, fontweight="bold")
ax.text(B[0] + 0.1, B[1] - 0.3, "B=(4,0)", color=PURP, fontsize=12, fontweight="bold")
ax.text(proj[0] - 0.3, proj[1] - 0.4, "proj", color="#b9821e", fontsize=11, fontweight="bold")
ax.set_title(f"cosine(A,B) = {cos:.2f},   |proj| = {proj_len:.1f}", color=INK, fontsize=13, fontweight="bold")
ax.set_xlim(-1, 5); ax.set_ylim(-1, 5); ax.set_aspect("equal")
ax.axhline(0, color="#888", lw=0.8); ax.axvline(0, color="#888", lw=0.8); ax.grid(alpha=0.15)
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white", bbox_inches="tight")
print(f"saved {FIG.name}  (cos={cos:.2f}, |proj|={proj_len:.1f})")


In [ ]:
%%writefile ep10_eigen.py
"""
M10 · Eigenvalues & Eigenvectors — invariant directions: A v = lambda v.
For a symmetric (covariance-like) matrix we find the directions A only stretches, never
turns. We take the top eigenvector and verify A v lands exactly on lambda * v. Seeded, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m10.png"

rng = np.random.default_rng(10)
A = np.array([[2.0, 1.0],
              [1.0, 2.0]])                       # b1: symmetric, covariance-like

vals, vecs = np.linalg.eigh(A)                   # b2: symmetric -> eigh (ascending)
order = np.argsort(vals)[::-1]
vals, vecs = vals[order], vecs[:, order]
v = vecs[:, 0]
if v[0] < 0:                                      # fix sign so the top eigenvector points + (display)
    v = -v
print("eigenvalues:", np.round(vals, 4))

Av = A @ v                                        # b3: apply A to the top eigenvector
print("top eigenvector v:", np.round(v, 4))
print("A v             :", np.round(Av, 4))
print("lambda * v      :", np.round(vals[0] * v, 4))

res = float(np.linalg.norm(Av - vals[0] * v))     # b4: residual ~ 0
print("residual ||Av - lambda v|| =", round(res, 6))

# b5: figure — v and A v lie on one line
BLUE, INK = "#7cc1ff", "#1b2330"
fig, ax = plt.subplots(figsize=(5.2, 5.2), dpi=130)
ax.set_title(f"A v = lambda v   (lambda1 = {vals[0]:.1f})", color=INK, fontsize=13, fontweight="bold")
ax.annotate("", xy=Av, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=BLUE, lw=5))
ax.annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="#2a6fb0", lw=2.4))
ax.text(v[0] - 0.18, v[1] + 0.12, "v", color="#2a6fb0", fontsize=14, fontweight="bold")
ax.text(Av[0] + 0.06, Av[1] - 0.02, "A v = 3v", color=BLUE, fontsize=13, fontweight="bold")
ax.axhline(0, color="#888", lw=0.8); ax.axvline(0, color="#888", lw=0.8)
ax.set_xlim(-0.5, 2.7); ax.set_ylim(-0.5, 2.7); ax.set_aspect("equal"); ax.grid(alpha=0.18)
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white", bbox_inches="tight")
print(f"saved {FIG.name}  (lambda1={vals[0]:.1f}, residual={res:.1e})")


In [ ]:
%%writefile ep11_svd.py
"""
M11 · Singular Value Decomposition — A = U S V^T for ANY matrix (rotate, stretch, rotate).
We take a near-low-rank 6x4 matrix, run SVD, rebuild it from only the top-2 singular values,
and measure the energy kept. Seeded, offline. Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m11.png"

rng = np.random.default_rng(11)
base = np.outer([3, 3, 2, 2, 1, 1], [1.0, 0.8, 0.6, 0.4])   # b1: dominant low-rank pattern
A = base + 0.05 * rng.standard_normal((6, 4))               #     + small noise

U, s, Vt = np.linalg.svd(A, full_matrices=False)             # b2: SVD
print("singular values:", np.round(s, 3))

k = 2                                                        # b3: rank-k reconstruction
Ak = (U[:, :k] * s[:k]) @ Vt[:k]
err = float(np.linalg.norm(A - Ak) / np.linalg.norm(A))
print("rank-2 relative error:", round(err, 4))

energy = float(np.sum(s[:k] ** 2) / np.sum(s ** 2))          # b4: energy captured
print("energy captured (k=2):", round(energy, 4))

# b5: figure — singular-value spectrum, kept vs dropped
BLUE, DIM, INK = "#7cc1ff", "#33414f", "#1b2330"
fig, ax = plt.subplots(figsize=(5.6, 4.2), dpi=130)
colors = [BLUE if i < k else DIM for i in range(len(s))]
bars = ax.bar(range(1, len(s) + 1), s, color=colors, edgecolor="#1b2330", lw=0.6)
for i, (b, v) in enumerate(zip(bars, s)):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.12, f"{v:.2f}", ha="center",
            fontsize=10, fontweight="bold", color=BLUE if i < k else "#7d8693")
ax.text(1.5, max(s) * 0.55, "kept (k=2)", color=BLUE, fontsize=11, fontweight="bold")
ax.text(3.5, max(s) * 0.2, "dropped", color="#7d8693", fontsize=10)
ax.set_title(f"Singular values  -  energy(k=2) = {energy:.4f}", color=INK, fontsize=12, fontweight="bold")
ax.set_xlabel("index"); ax.set_ylabel("singular value")
ax.set_xticks(range(1, len(s) + 1)); ax.grid(alpha=0.18, axis="y")
for sp in ["top", "right"]:
    ax.spines[sp].set_visible(False)
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white", bbox_inches="tight")
print(f"saved {FIG.name}  (s1={s[0]:.2f}, energy={energy:.4f}, err={err:.4f})")


In [ ]:
%%writefile ep12_pca.py
"""
M12 · Principal Component Analysis — covariance eigenvectors are the principal axes.
We make a tilted, correlated 2D cloud, center it, build the covariance matrix, and read its
eigenvectors as the directions of greatest spread (the principal components). Seeded, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m12.png"

rng = np.random.default_rng(12)
X = rng.standard_normal((300, 2)) * [3.0, 1.0]               # b1: stretch isotropic noise
theta = np.deg2rad(30)
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
X = X @ R.T                                                  #     then rotate 30 degrees

Xc = X - X.mean(0)                                           # b2: center + covariance
C = np.cov(Xc, rowvar=False)

vals, vecs = np.linalg.eigh(C)                               # b3: eigendecompose (descending)
order = np.argsort(vals)[::-1]
vals, vecs = vals[order], vecs[:, order]
if vecs[0, 0] < 0:                                            # sign-fix PC1 for display determinism
    vecs[:, 0] = -vecs[:, 0]

ratio = vals / vals.sum()                                    # b4: variance explained
print("eigenvalues:", np.round(vals, 3))
print("variance explained:", np.round(ratio, 3))
print("PC1 direction:", np.round(vecs[:, 0], 3))

# b5: figure — cloud with principal axes scaled by sqrt(variance)
BLUE, GOLD, INK = "#7cc1ff", "#ffd166", "#1b2330"
fig, ax = plt.subplots(figsize=(5.4, 5.0), dpi=130)
ax.set_title(f"PCA  -  PC1 explains {ratio[0]:.1%}", color=INK, fontsize=13, fontweight="bold")
ax.scatter(Xc[:, 0], Xc[:, 1], s=9, color=BLUE, alpha=0.45)
for i, (w, lab) in enumerate([(3.2, "PC1"), (2.2, "PC2")]):
    d = vecs[:, i] * np.sqrt(vals[i]) * 2.4
    ax.annotate("", xy=d, xytext=(-d[0], -d[1]), arrowprops=dict(arrowstyle="-|>", color=GOLD, lw=w))
    ax.text(d[0], d[1], f" {lab} ({ratio[i]:.0%})", color="#b9821e", fontsize=11, fontweight="bold")
ax.axhline(0, color="#bbb", lw=0.7); ax.axvline(0, color="#bbb", lw=0.7)
ax.set_aspect("equal"); ax.grid(alpha=0.18)
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, facecolor="white", bbox_inches="tight")
print(f"saved {FIG.name}  (PC1 {ratio[0]:.1%}, PC1 dir {np.round(vecs[:,0],3)})")


In [ ]:
%%writefile ep13_tensors.py
"""
M13 · Tensors & Broadcasting — a tensor is an n-D array; its shape is the contract.
Broadcasting lets a small array act stretched to match a big one: a length-3 bias adds to a
whole (4,3) batch with no loop and no copy. We verify every row got the same bias. Seeded, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m13.png"

rng = np.random.default_rng(13)
X = np.round(rng.uniform(0, 9, size=(4, 3)), 1)      # b1: batch — 4 samples, 3 features
bias = np.array([1.0, 2.0, 3.0])                     # b2: one bias per feature, shape (3,)

Y = X + bias                                         # b3: broadcast (4,3) + (3,) -> (4,3)
col = X.reshape(12, 1)                               # b4: reshape is free relabeling
back = col.reshape(4, 3)
same = bool(np.allclose((Y - X), np.tile(bias, (4, 1))))   # b5: every row got the same bias

print("X.shape    =", X.shape)
print("bias.shape =", bias.shape)
print("Y.shape    =", Y.shape)
print("X row0     =", X[0])
print("Y row0     =", Y[0])
print("reshape ok =", bool(np.array_equal(back, X)))
print("same bias all rows =", same)

# hero figure: Y - X is exactly the bias on every row
BLUE, INK = "#7cc1ff", "#1b2330"
fig, ax = plt.subplots(figsize=(6.0, 4.2), dpi=130); fig.patch.set_facecolor("white")
ax.imshow(Y - X, cmap="Blues", aspect="auto", vmin=0, vmax=3.6)
for i in range(4):
    for j in range(3):
        ax.text(j, i, f"{(Y - X)[i, j]:.1f}", ha="center", va="center", color="#07090f", fontsize=14, fontweight="bold")
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["f0", "f1", "f2"]); ax.set_yticks(range(4))
ax.set_yticklabels([f"row {i}" for i in range(4)])
ax.set_title("Y - X = bias broadcast to every row  (1.0, 2.0, 3.0)", color=INK, fontsize=12, fontweight="bold")
plt.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG, dpi=130, facecolor="white")
print(f"saved {FIG.name}  (Y-X every row = bias, same_all={same})")


In [ ]:
%%writefile ep14_derivatives.py
"""
M14 · Derivatives & Limits — a derivative is the slope of a curve at one point.
We approximate the slope of x^2 at x=1.5 with a shrinking secant (rise over run),
compare it to the exact power-rule value 2x = 3.0, and plot the tangent we found.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m14.png"

f = lambda x: x**2                 # b1: the function
df = lambda x: 2 * x               # exact derivative (power rule)

x0 = 1.5                           # b2: point of interest
h = 1e-5                           # tiny step

num = (f(x0 + h) - f(x0)) / h      # b3: secant slope -> numerical derivative
exact = df(x0)                     # b4: 2 * 1.5 = 3.0
err = abs(num - exact)

print("x0           =", x0)
print("numerical f' =", round(num, 5))
print("exact f'     =", exact)
print("abs error    =", f"{err:.2e}")

xs = np.linspace(0, 3, 200)        # b5: curve + tangent at x0
tangent = f(x0) + num * (xs - x0)

GREEN, INK = "#5CE08A", "#1b2330"
fig, ax = plt.subplots(figsize=(6.0, 4.2), dpi=130); fig.patch.set_facecolor("white")
ax.plot(xs, f(xs), color="#222", lw=2, label="f(x) = x^2")
ax.plot(xs, tangent, color=GREEN, lw=2.4, label=f"tangent slope = {num:.2f}")
ax.scatter([x0], [f(x0)], color=GREEN, s=70, zorder=5)
ax.annotate("x = 1.5", (x0, f(x0)), textcoords="offset points", xytext=(10, -16),
            color=INK, fontsize=11, fontweight="bold")
ax.set_title(f"Tangent at x=1.5: slope = {num:.2f}", color=GREEN, fontsize=13, fontweight="bold")
ax.set_xlabel("x"); ax.set_ylabel("f(x)"); ax.legend(loc="upper left", fontsize=10)
ax.grid(alpha=0.15)
plt.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG, dpi=130, facecolor="white")
print(f"saved {FIG.name}  (numerical {num:.5f} vs exact {exact}, err {err:.1e})")


In [ ]:
%%writefile ep15_gradients.py
"""
M15 · Multivariable Calculus & Gradients — when a function has many inputs, each input
has its own slope (a partial derivative). Stack the partials into a vector and you get the
gradient: it points in the direction of steepest ASCENT. Here: gradient of the bowl
f(x,y)=x^2+y^2 at (1,2), checked numerically, drawn on a contour map.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m15.png"

f = lambda x, y: x**2 + y**2                  # b1: bowl-shaped loss
grad = lambda x, y: np.array([2 * x, 2 * y])  # b2: exact gradient

p = np.array([1.0, 2.0])                      # b3: point of interest
g = grad(*p)                                  # exact gradient there -> [2, 4]

h = 1e-5                                       # b4: numerical check of each partial
gx = (f(p[0] + h, p[1]) - f(*p)) / h
gy = (f(p[0], p[1] + h) - f(*p)) / h
gnorm = float(np.linalg.norm(g))

print("point        =", tuple(float(v) for v in p))
print("exact grad   =", tuple(float(v) for v in g))
print("numeric grad = (%.3f, %.3f)" % (gx, gy))
print("|grad|       =", round(gnorm, 4))

xs = np.linspace(-3, 3, 200)                   # b5: contour rings + gradient arrow
X, Y = np.meshgrid(xs, xs)
GREEN = "#5CE08A"
fig, ax = plt.subplots(figsize=(5.6, 5.0), dpi=130); fig.patch.set_facecolor("white")
ax.contour(X, Y, f(X, Y), levels=10, colors="#cfd8e3", linewidths=1.1)
ax.quiver(p[0], p[1], g[0], g[1], color=GREEN, angles="xy", scale_units="xy", scale=1.0, width=0.012, zorder=4)
ax.scatter(*p, color=GREEN, s=70, zorder=5)
ax.annotate("(1, 2)", (p[0], p[1]), textcoords="offset points", xytext=(-38, -4),
            color="#1b2330", fontsize=11, fontweight="bold")
ax.set_title(f"grad f at (1,2) = (2,4),  |grad|={gnorm:.2f}", color=GREEN, fontsize=12, fontweight="bold")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_aspect("equal"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
plt.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG, dpi=130, facecolor="white")
print(f"saved {FIG.name}  (grad (2,4) outward, perpendicular to rings, |grad|={gnorm:.4f})")


In [ ]:
%%writefile ep16_chainrule.py
"""
M16 · The Chain Rule — the derivative of a nested function f(g(x)) is the product of the
derivatives at each link: dy/dx = f'(g(x)) * g'(x). Here g(x)=3x+1, f(u)=u^2, so the
composite is (3x+1)^2; by hand dy/dx = 2(3x+1)*3 = 6(3x+1), which is 42 at x=2.
We cross-check against a central finite difference. Seeded, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m16.png"

g = lambda x: 3.0 * x + 1.0           # inner
f = lambda u: u ** 2                  # outer
gp = lambda x: 3.0                    # g'
fp = lambda u: 2.0 * u                # f'

def chain(x):                          # dy/dx = f'(g(x)) * g'(x)
    return fp(g(x)) * gp(x)

def numeric(x, eps=1e-6):
    h = lambda t: f(g(t))
    return (h(x + eps) - h(x - eps)) / (2 * eps)

x0 = 2.0
print(f"g(x0)      = {g(x0):.4f}")
print(f"chain dy/dx= {chain(x0):.4f}")
print(f"numeric    = {numeric(x0):.4f}")
print(f"match       = {bool(np.isclose(chain(x0), numeric(x0)))}")

xs = np.linspace(-2, 4, 60)
GREEN = "#5CE08A"
fig, ax = plt.subplots(figsize=(6.0, 4.2), dpi=130); fig.patch.set_facecolor("white")
ax.plot(xs, chain(xs), color=GREEN, lw=3, label="chain rule  6(3x+1)")
ax.plot(xs, numeric(xs), "k--", lw=1.6, label="numeric")
ax.scatter([x0], [chain(x0)], color=GREEN, s=70, zorder=5)
ax.annotate("dy/dx@2 = 42", (x0, chain(x0)), textcoords="offset points", xytext=(-120, -6),
            color="#1b2330", fontsize=11, fontweight="bold")
ax.set_title("Chain rule slope of (3x+1)^2   (dy/dx@2 = 42)", color=GREEN, fontsize=12, fontweight="bold")
ax.set_xlabel("x"); ax.set_ylabel("dy/dx"); ax.legend(loc="upper left", fontsize=10); ax.grid(alpha=0.15)
plt.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG, dpi=130, facecolor="white")
print(f"saved {FIG.name}  (chain == numeric, dy/dx@2 = {chain(x0):.1f})")


In [ ]:
%%writefile ep17_autodiff.py
"""
M17 · Computational Graphs & Automatic Differentiation — represent z = x*y + sin(x) as a
graph, run a forward pass, then let PyTorch's reverse-mode autodiff (.backward()) sweep the
graph once to get dz/dx and dz/dy. We cross-check against the hand derivatives
dz/dx = y + cos(x), dz/dy = x. Seeded, offline.
Runnable: pip install torch matplotlib
"""
import pathlib
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m17.png"

torch.manual_seed(0)
x = torch.tensor(1.5, requires_grad=True)   # b1: leaf nodes record the graph
y = torch.tensor(2.0, requires_grad=True)

z = x * y + torch.sin(x)                      # b2: forward pass builds the graph
z.backward()                                  # b3: reverse-mode autodiff fills .grad

dzdx_auto = x.grad.item()
dzdy_auto = y.grad.item()
dzdx_hand = (y + torch.cos(x)).item()         # b4: dz/dx = y + cos(x)
dzdy_hand = x.item()                          # dz/dy = x

print(f"z              = {z.item():.4f}")
print(f"dz/dx autograd = {dzdx_auto:.4f}   hand = {dzdx_hand:.4f}")
print(f"dz/dy autograd = {dzdy_auto:.4f}   hand = {dzdy_hand:.4f}")
print(f"match           = {bool(abs(dzdx_auto - dzdx_hand) < 1e-5 and abs(dzdy_auto - dzdy_hand) < 1e-5)}")

labels = ["dz/dx", "dz/dy"]                    # b5: paired bars, autograd vs hand
auto = [dzdx_auto, dzdy_auto]; hand = [dzdx_hand, dzdy_hand]
xpos = range(len(labels))
GREEN = "#5CE08A"
fig, ax = plt.subplots(figsize=(6.0, 4.2), dpi=130); fig.patch.set_facecolor("white")
ax.bar([p - 0.18 for p in xpos], auto, 0.36, color=GREEN, label="autograd")
ax.bar([p + 0.18 for p in xpos], hand, 0.36, color="#9aa3b2", label="hand")
for p, v in zip(xpos, auto):
    ax.text(p - 0.18, v + 0.03, f"{v:.3f}", ha="center", fontsize=10, fontweight="bold", color="#1b2330")
ax.set_xticks(list(xpos)); ax.set_xticklabels(labels, fontsize=12)
ax.set_title(f"Autograd vs hand gradients  (dz/dx={dzdx_auto:.3f}, dz/dy={dzdy_auto:.3f})", color=GREEN, fontsize=12, fontweight="bold")
ax.legend(loc="upper right", fontsize=10); ax.grid(axis="y", alpha=0.15)
plt.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG, dpi=130, facecolor="white")
print(f"saved {FIG.name}  (autograd == hand: dz/dx={dzdx_auto:.4f}, dz/dy={dzdy_auto:.4f})")


In [ ]:
%%writefile ep18_backprop.py
"""
M18 · Backpropagation Mathematics — derive backprop for a tiny 2-2-1 MLP BY HAND and check
every weight gradient against PyTorch autograd. Forward: z1 = W1 x + b1, h = sigmoid(z1),
pred = W2 h + b2, loss = MSE. Backward: d_out = 2(pred - t); gW2 = outer(d_out, h);
d_h = (W2^T d_out) * h(1-h); gW1 = outer(d_h, x). Seeded, offline.
Runnable: pip install torch matplotlib
"""
import pathlib
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m18.png"

torch.manual_seed(0)
sig = torch.sigmoid
x = torch.tensor([1.0, -2.0])                 # b1: fixed input + target + weights
t = torch.tensor([0.5])
W1 = torch.tensor([[0.1, 0.2], [-0.3, 0.4]], requires_grad=True)
b1 = torch.tensor([0.0, 0.0], requires_grad=True)
W2 = torch.tensor([[0.5, -0.6]], requires_grad=True)
b2 = torch.tensor([0.1], requires_grad=True)

z1 = W1 @ x + b1; h = sig(z1)                  # b2: forward
pred = W2 @ h + b2; loss = 0.5 * ((pred - t) ** 2).mean()   # 1/2 MSE -> clean delta

d_out = (pred - t)                              # b3: backward by hand; d_out = pred - target
gW2 = torch.outer(d_out, h)
d_h = (W2.t() @ d_out) * h * (1 - h)            # sigmoid'(z1) = h(1-h)
gW1 = torch.outer(d_h, x)                       # b4: W1 grad = d_h outer x

loss.backward()                                  # b5: autograd check
gW2_hand = gW2.detach().numpy().round(4).ravel()
gW2_auto = W2.grad.numpy().round(4).ravel()
gW1_hand = gW1.detach().numpy().round(4).ravel()
gW1_auto = W1.grad.numpy().round(4).ravel()
match = bool(
    torch.allclose(gW2.detach(), W2.grad, atol=1e-6) and torch.allclose(gW1.detach(), W1.grad, atol=1e-6)
)
print(f"loss        = {loss.item():.4f}")
print(f"gW2 hand    = {gW2_hand}")
print(f"gW2 autograd= {gW2_auto}")
print(f"gW1 hand    = {gW1_hand}")
print(f"gW1 autograd= {gW1_auto}")
print(f"match        = {match}")

hand = torch.cat([gW2.detach().ravel(), gW1.detach().ravel()]).numpy()
auto = torch.cat([W2.grad.ravel(), W1.grad.ravel()]).numpy()
labels = ["gW2[0]", "gW2[1]", "gW1[0]", "gW1[1]", "gW1[2]", "gW1[3]"]
p = range(len(hand))
GREEN = "#5CE08A"
fig, ax = plt.subplots(figsize=(6.2, 4.2), dpi=130); fig.patch.set_facecolor("white")
ax.bar([i - 0.18 for i in p], hand, 0.36, color=GREEN, label="hand")
ax.bar([i + 0.18 for i in p], auto, 0.36, color="#9aa3b2", label="autograd")
ax.axhline(0, color="#cfd8e3", lw=1)
ax.set_xticks(list(p)); ax.set_xticklabels(labels, fontsize=9)
ax.set_title(f"Backprop: hand vs autograd weight grads (loss={loss.item():.4f})", color=GREEN, fontsize=12, fontweight="bold")
ax.legend(loc="upper right", fontsize=10); ax.grid(axis="y", alpha=0.15)
plt.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG, dpi=130, facecolor="white")
print(f"saved {FIG.name}  (hand == autograd: {match})")


In [ ]:
%%writefile ep19_gradient_descent.py
"""
M19 · Gradient Descent & its Variants — the one rule behind all training:
new = old - learning_rate * gradient. We minimize a quadratic bowl f(x)=x^2 (min at x=0),
start far away at x=5, and watch plain gradient descent walk to the bottom. Each step
multiplies x by (1 - eta*2) = 0.8, so the loss collapses geometrically toward zero. Seeded, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m19.png"

rng = np.random.default_rng(19)          # b1: seed (reproducible)
f = lambda x: x ** 2                      # b1: loss is a bowl, min at x=0
grad = lambda x: 2.0 * x                  # b2: derivative 2x

x = 5.0                                   # b3: start far from the minimum
eta = 0.1                                 # b3: learning rate
xs = [x]
for _ in range(40):                       # b3: descend
    x = x - eta * grad(x)                 #     new = old - eta * gradient
    xs.append(x)
xs = np.array(xs)
losses = f(xs)

print(f"start x = {xs[0]:.4f}  loss = {losses[0]:.4f}")
print(f"step  1 x = {xs[1]:.4f}  loss = {losses[1]:.4f}")
print(f"step  5 x = {xs[5]:.4f}  loss = {losses[5]:.4f}")
print(f"final x = {xs[-1]:.4f}  loss = {losses[-1]:.6f}")

GREEN = "#5CE08A"
fig, ax = plt.subplots(figsize=(6.0, 4.2), dpi=130); fig.patch.set_facecolor("white")
ax.plot(range(len(losses)), losses, "-o", color=GREEN, ms=4, lw=2)
ax.set_title("Gradient descent: loss -> 0", color=GREEN, fontsize=13, fontweight="bold")
ax.set_xlabel("step"); ax.set_ylabel("loss = x^2"); ax.grid(alpha=0.15)
ax.annotate(f"start = {losses[0]:.0f}", (0, losses[0]), textcoords="offset points", xytext=(14, -6), color="#1b2330", fontsize=10, fontweight="bold")
ax.annotate("loss -> 0", (len(losses) - 1, losses[-1]), textcoords="offset points", xytext=(-70, 18), color=GREEN, fontsize=10, fontweight="bold")
plt.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG, dpi=130, facecolor="white")
print(f"saved {FIG.name}  (start loss {losses[0]:.1f} -> final {losses[-1]:.6f})")


In [ ]:
%%writefile ep20_convexity.py
"""
M20 · Convex Optimization & Numerical Methods — a convex loss has ONE valley (any honest
downhill method finds the single global minimum), a nonconvex loss has many. We run the SAME
gradient descent on a nonconvex curve f(x)=x^2+10*sin(x) from two different starts and watch
them land in DIFFERENT valleys — nonconvexity in one number. Seeded, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m20.png"

rng = np.random.default_rng(20)                       # b1: seed
convex = lambda x: x ** 2                             # b1: one valley
nonconvex = lambda x: x ** 2 + 10 * np.sin(x)        # b1: several valleys
dnc = lambda x: 2 * x + 10 * np.cos(x)               # b1: its gradient

grid = np.linspace(-8, 8, 400)                        # b2: draw shapes
void_c = convex(grid)                                 # (drawn in exercises; kept for parity)

def descend(x0, eta=0.02, steps=300):                 # b3: roll downhill
    x = x0
    for _ in range(steps):
        x = x - eta * dnc(x)
    return x, nonconvex(x)

xa, la = descend(-6.0)                                # b4: start left
xb, lb = descend(4.0)                                 # b4: start right
print(f"start -6.0 -> x = {xa:.3f}  loss = {la:.3f}")
print(f"start  4.0 -> x = {xb:.3f}  loss = {lb:.3f}")
print(f"same algorithm, different minima: {abs(la - lb):.3f} apart")

GREEN, GOLD = "#5CE08A", "#ffd166"
fig, ax = plt.subplots(figsize=(6.0, 4.2), dpi=130); fig.patch.set_facecolor("white")
ax.plot(grid, nonconvex(grid), color=GREEN, lw=2, label="nonconvex loss")
ax.scatter([xa, xb], [la, lb], color=GOLD, s=80, zorder=5, label="GD endpoints")
ax.annotate(f"{la:.2f}", (xa, la), textcoords="offset points", xytext=(8, -14), color="#1b2330", fontsize=10, fontweight="bold")
ax.annotate(f"{lb:.2f}", (xb, lb), textcoords="offset points", xytext=(8, 8), color="#1b2330", fontsize=10, fontweight="bold")
ax.set_title("Nonconvex: start decides the minimum", color=GREEN, fontsize=12, fontweight="bold")
ax.set_xlabel("x"); ax.set_ylabel("loss"); ax.legend(loc="upper center", fontsize=9); ax.grid(alpha=0.15)
plt.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG, dpi=130, facecolor="white")
print(f"saved {FIG.name}  (two starts -> losses {la:.3f} vs {lb:.3f})")


In [ ]:
%%writefile ep21_probability.py
"""
M21 · Probability & Random Variables — probability is the math of uncertainty: list the
outcomes, weight each between 0 and 1 so they sum to one. A random variable attaches a number
to each outcome; its EXPECTATION is the probability-weighted average and its VARIANCE the
average squared spread. We build a loaded six-sided die, compute E[X] and Var[X] from the
definitions, and confirm them by 100k simulated rolls (law of large numbers). Seeded, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m21.png"

rng = np.random.default_rng(21)                          # b1: seed
vals = np.array([1, 2, 3, 4, 5, 6])                     # b1: outcomes
p = np.array([0.05, 0.05, 0.10, 0.15, 0.25, 0.40])     # b1: loaded, sums to 1

assert np.all(p >= 0) and abs(p.sum() - 1) < 1e-9       # b2: axioms hold
E = (vals * p).sum()                                     # b3: expectation
Var = ((vals - E) ** 2 * p).sum()                        # b3: variance

rolls = rng.choice(vals, size=100_000, p=p)             # b4: simulate
mE, mV = rolls.mean(), rolls.var()                       # b4: sample stats

print(f"theory   E[X] = {E:.4f}   Var[X] = {Var:.4f}")
print(f"sampled  mean = {mE:.4f}   var    = {mV:.4f}")
print(f"agree to ~2 decimals: {abs(E - mE):.3f}, {abs(Var - mV):.3f}")

VIOLET, GOLD = "#A98BFF", "#ffd166"
fig, ax = plt.subplots(figsize=(6.0, 4.2), dpi=130); fig.patch.set_facecolor("white")
ax.bar(vals, p, color=VIOLET, label="P(X)", width=0.7)
ax.axvline(E, color=GOLD, lw=2.5, label=f"E[X] = {E:.2f}")
ax.set_title("Loaded die: probabilities and mean", color=VIOLET, fontsize=13, fontweight="bold")
ax.set_xlabel("face"); ax.set_ylabel("P"); ax.set_xticks(vals); ax.legend(loc="upper left", fontsize=10); ax.grid(axis="y", alpha=0.15)
plt.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG, dpi=130, facecolor="white")
print(f"saved {FIG.name}  (E[X]={E:.4f}, Var={Var:.4f}; sampled {mE:.4f}/{mV:.4f})")


In [ ]:
%%writefile ep22_distributions.py
"""
M22 · Probability Distributions — a distribution spreads a total of 1.0 across outcomes.
Discrete -> a PMF (probability sits on points); continuous -> a PDF (probability is AREA under
a smooth curve). We sample a Bernoulli (biased coin), a Gaussian (standard normal), and turn
raw logits into a Categorical with softmax, then plot all three. Seeded, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m22.png"

rng = np.random.default_rng(22)                       # b0/b1: reproducible generator

bern = (rng.random(10000) < 0.3)                      # b2: Bernoulli(p=0.3), a biased coin
p_hat = bern.mean()

gauss = rng.normal(0.0, 1.0, 10000)                   # b3: Gaussian(0,1), standard normal
g_mean, g_std = gauss.mean(), gauss.std()

logits = np.array([2.0, 1.0, 0.1])                    # b4: softmax -> categorical
cat = np.exp(logits) / np.exp(logits).sum()

print(f"Bernoulli p_hat = {p_hat:.4f}  (true 0.3)")
print(f"Gaussian  mean  = {g_mean:.4f}  std = {g_std:.4f}")
print(f"Softmax   probs = {np.round(cat, 4)}  sum = {cat.sum():.4f}")

V = "#A98BFF"
fig, ax = plt.subplots(1, 3, figsize=(11, 3.6), dpi=120); fig.patch.set_facecolor("white")
ax[0].bar([0, 1], [1 - p_hat, p_hat], color=V); ax[0].set_title(f"Bernoulli p_hat={p_hat:.4f}", color="#1b2330", fontsize=11)
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(["0", "1"])
ax[1].hist(gauss, bins=40, color=V, density=True); ax[1].set_title(f"Gaussian m={g_mean:.4f} s={g_std:.4f}", color="#1b2330", fontsize=11)
ax[2].bar([0, 1, 2], cat, color=V); ax[2].set_title(f"Categorical sum={cat.sum():.4f}", color="#1b2330", fontsize=11)
ax[2].set_xticks([0, 1, 2])
fig.suptitle("Bernoulli  |  Gaussian  |  Categorical", color=V, fontsize=14, fontweight="bold")
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, dpi=120, facecolor="white")
print(f"saved {FIG.name}  (p_hat={p_hat:.4f}, gauss {g_mean:.4f}/{g_std:.4f}, cat {np.round(cat,4)})")


In [ ]:
%%writefile ep23_bayes.py
"""
M23 · Bayes' Theorem — the math of changing your mind: posterior = likelihood * prior / evidence.
The classic base-rate trap: a 99%-accurate test for a 1%-common disease still leaves you only
~1-in-6 likely to be sick after one positive, because false alarms on the huge healthy crowd
swamp the few true positives. Exact, deterministic, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m23.png"

p_disease = 0.01                       # b1: prior -- how common the condition is
sens = 0.99                            # b2: P(+ | disease)  (true-positive rate)
fpr = 0.05                             # b2: P(+ | no disease) (false-positive rate)

p_pos = sens * p_disease + fpr * (1 - p_disease)   # b3: evidence -- total P(positive)
posterior = sens * p_disease / p_pos               # b4: P(disease | +)

true_pos = sens * p_disease                          # ingredients of the evidence
false_pos = fpr * (1 - p_disease)

print(f"Prior  P(disease)      = {p_disease:.4f}")
print(f"Evidence P(positive)   = {p_pos:.4f}")
print(f"Posterior P(disease|+) = {posterior:.4f}")

V, DIM = "#A98BFF", "#5b4a82"
fig, ax = plt.subplots(figsize=(6.2, 4.2), dpi=130); fig.patch.set_facecolor("white")
ax.bar(["true +", "false +"], [true_pos, false_pos], color=[V, DIM], width=0.6)
ax.text(0, true_pos + 0.002, f"{true_pos:.4f}", ha="center", fontsize=10, fontweight="bold", color="#1b2330")
ax.text(1, false_pos + 0.002, f"{false_pos:.4f}", ha="center", fontsize=10, fontweight="bold", color="#1b2330")
ax.set_title(f"P(disease|+) = {posterior:.4f}   (P(+) = {p_pos:.4f})", color=V, fontsize=12, fontweight="bold")
ax.set_ylabel("probability mass"); ax.grid(axis="y", alpha=0.15)
plt.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, dpi=130, facecolor="white")
print(f"saved {FIG.name}  (P(+)={p_pos:.4f}, posterior={posterior:.4f}; true+ {true_pos:.4f} vs false+ {false_pos:.4f})")


In [ ]:
%%writefile ep24_statistics.py
"""
M24 · Statistics, Sampling & Estimation — you never see the whole population, only a sample,
and you guess the truth from it. Two laws make that trustworthy: the Law of Large Numbers
(the running sample mean converges to the true mean) and the Central Limit Theorem (the spread
of sample means shrinks like sigma/sqrt(n) and goes bell-shaped). Seeded, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m24.png"

mu, sigma = 5.0, 2.0                              # b1: known population
rng = np.random.default_rng(24)

x = rng.normal(mu, sigma, 2000)                  # b2: Law of Large Numbers — running mean
running = np.cumsum(x) / np.arange(1, 2001)

n, reps = 30, 5000                               # b3: sampling distribution of the mean
means = rng.normal(mu, sigma, (reps, n)).mean(axis=1)

se_theory = sigma / np.sqrt(n)                   # b4: standard error = sigma/sqrt(n)

print(f"Running mean (2000 draws) = {running[-1]:.4f}  (true {mu})")
print(f"Mean of sample-means      = {means.mean():.4f}")
print(f"Std of sample-means       = {means.std():.4f}  (theory {se_theory:.4f})")

V = "#A98BFF"
fig, ax = plt.subplots(1, 2, figsize=(10.4, 3.8), dpi=130); fig.patch.set_facecolor("white")
ax[0].plot(running, color=V, lw=1.6); ax[0].axhline(mu, ls="--", color="#9aa3b2")
ax[0].set_title(f"running mean -> {running[-1]:.4f}  (true 5)", color=V, fontsize=12, fontweight="bold")
ax[0].set_xlabel("samples seen"); ax[0].set_ylabel("running mean"); ax[0].grid(alpha=0.15)
ax[1].hist(means, bins=40, color=V, density=True, alpha=0.85)
ax[1].axvline(mu, ls="--", color="#9aa3b2")
ax[1].set_title(f"sample-means std={means.std():.4f}  (theory {se_theory:.4f})", color=V, fontsize=12, fontweight="bold")
ax[1].set_xlabel("sample mean (n=30)"); ax[1].grid(alpha=0.15)
fig.suptitle("Law of Large Numbers + Central Limit Theorem", color=V, fontsize=14, fontweight="bold")
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, dpi=130, facecolor="white")
print(f"saved {FIG.name}  (running {running[-1]:.4f}, means {means.mean():.4f}/{means.std():.4f} vs theory {se_theory:.4f})")


In [ ]:
%%writefile ep25_hypothesis.py
"""
M25 · Hypothesis Testing — there is always SOME gap between two models; the real question is
"would noise this big appear by accident if they were truly identical?" Assume the null (no real
difference), measure the t-statistic (gap / standard error), turn it into a p-value (chance of a
gap this big under the null), and reject the null only when p is small. This is A/B testing.
Seeded, offline.
Runnable: pip install numpy scipy matplotlib
"""
import pathlib
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m25.png"

rng = np.random.default_rng(25)                 # b1: reproducible draws
a = rng.normal(0.80, 0.05, 50)                  # model A per-example scores
b = rng.normal(0.84, 0.05, 50)                  # model B per-example scores

ma, mb = a.mean(), b.mean()                      # b2: means + pooled standard error
sp = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
se = sp * np.sqrt(2 / 50)
t = (mb - ma) / se                               # b3: t-statistic
df = 2 * 50 - 2
p = 2 * stats.t.sf(abs(t), df)                   # b4: two-sided p-value
tc = stats.t.ppf(0.975, df)
ci = ((mb - ma) - tc * se, (mb - ma) + tc * se)  # 95% CI for the true gap

print(f"mean A = {ma:.4f}  mean B = {mb:.4f}  gap = {mb - ma:.4f}")
print(f"t = {t:.3f}  df = {df}  p = {p:.4f}")
print(f"95% CI for gap = [{ci[0]:.4f}, {ci[1]:.4f}]")
print("verdict:", "B is significantly better" if p < 0.05 else "no significant difference")

GREY, V = "#9aa3b2", "#A98BFF"
fig, ax = plt.subplots(figsize=(7.2, 4.2), dpi=130); fig.patch.set_facecolor("white")
ax.hist(a, bins=12, alpha=0.6, color=GREY, label=f"A  mean={ma:.3f}")
ax.hist(b, bins=12, alpha=0.75, color=V, label=f"B  mean={mb:.3f}")
ax.axvline(ma, color=GREY, lw=2); ax.axvline(mb, color=V, lw=2)
ax.set_title(f"two-sample t-test:  gap={mb - ma:.3f}  t={t:.2f}  p={p:.4f}", color=V, fontsize=12, fontweight="bold")
ax.set_xlabel("per-example score"); ax.set_ylabel("count"); ax.legend(loc="upper left", fontsize=10)
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, dpi=130, facecolor="white")
print(f"saved {FIG.name}  (gap {mb-ma:.4f}, t {t:.3f}, p {p:.4f})")


In [ ]:
%%writefile ep26_information.py
"""
M26 · Information Theory — Entropy, Cross-Entropy, KL.
Information is surprise. Entropy = average surprise of a distribution (its irreducible
uncertainty). Cross-entropy = the surprise you pay when you encode reality p with the
WRONG beliefs q. KL divergence = the overpayment, cross-entropy minus entropy, zero only
when q == p. Cross-entropy is THE classification / language-model loss; KL is the leash in
VAEs, distillation, and RLHF. Verifies CE = H + KL to machine precision. Seeded, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m26.png"

p = np.array([0.50, 0.25, 0.15, 0.10])           # b1: true distribution
q = np.array([0.40, 0.30, 0.20, 0.10])           # model's prediction
labels = ["cat", "dog", "bird", "fish"]

H  = -np.sum(p * np.log2(p))                      # b2: entropy of p (bits)
CE = -np.sum(p * np.log2(q))                      # b3: cross-entropy(p, q)
KL =  np.sum(p * np.log2(p / q))                  # b4: KL(p || q)

print(f"entropy H(p)        = {H:.4f} bits")
print(f"cross-entropy(p,q)  = {CE:.4f} bits")
print(f"KL(p || q)          = {KL:.4f} bits")
print(f"check  H + KL        = {H + KL:.4f} bits  (== cross-entropy)")
assert abs((H + KL) - CE) < 1e-9                  # CE = H + KL, exactly
print("identity CE = H + KL holds:", bool(abs((H + KL) - CE) < 1e-9))

GREY, V = "#7d8693", "#A98BFF"
x = np.arange(4)                                  # b5: figure == terminal
fig, ax = plt.subplots(figsize=(7, 4), dpi=120); fig.patch.set_facecolor("white")
ax.bar(x - 0.18, p, 0.36, color=V, label="true p")
ax.bar(x + 0.18, q, 0.36, color=GREY, label="model q")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("probability")
ax.set_title(f"H={H:.3f}   CE={CE:.3f}   KL={KL:.3f}  bits", color=V, fontweight="bold")
ax.legend(loc="upper right")
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, dpi=120, facecolor="white")
print(f"saved {FIG.name}  (H {H:.4f}, CE {CE:.4f}, KL {KL:.4f})")


In [ ]:
%%writefile ep27_regression.py
"""
M27 · The Math of Linear & Logistic Regression — the two atoms of supervised learning.
Linear regression fits a straight line by minimizing mean squared error: a smooth bowl in the
weights with a single lowest point, solvable in closed form (normal equations) OR by gradient
descent (they agree). Logistic regression wraps the same linear score w·x+b in a sigmoid to get
a probability, and trains with cross-entropy (last episode's loss); its decision boundary is the
line where the probability crosses 0.5 (w·x+b = 0). Seeded, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m27.png"

rng = np.random.default_rng(27)                          # b1: data
x = np.linspace(0, 5, 40); y = 1.8 * x + 0.5 + rng.normal(0, 0.6, 40)
X = np.c_[np.ones(40), x]                                # design matrix [1, x]

w_ne = np.linalg.solve(X.T @ X, X.T @ y)                # b2: normal equations
w = np.zeros(2)                                          # b3: gradient descent
for _ in range(8000):
    w -= 0.01 * (2 / 40) * X.T @ (X @ w - y)
print(f"normal eq:  intercept={w_ne[0]:.3f}  slope={w_ne[1]:.3f}")
print(f"grad desc:  intercept={w[0]:.3f}  slope={w[1]:.3f}")
print(f"weights match: {bool(np.allclose(w, w_ne, atol=1e-3))}")

g0 = rng.normal([1, 1], 0.6, (50, 2)); g1 = rng.normal([3, 3], 0.6, (50, 2))
P = np.c_[np.ones(100), np.vstack([g0, g1])]; t = np.r_[np.zeros(50), np.ones(50)]
wl = np.zeros(3)                                         # b4: logistic GD on cross-entropy
for _ in range(5000):
    pr = 1 / (1 + np.exp(-(P @ wl)))
    wl -= 0.1 * (P.T @ (pr - t)) / 100
pred = (1 / (1 + np.exp(-(P @ wl))) > 0.5).astype(int)
acc = (pred == t).mean()
print(f"logistic accuracy = {acc:.3f}")

CY, V, PK = "#34D6FF", "#A98BFF", "#ff6ec7"
fig, ax = plt.subplots(1, 2, figsize=(9, 4), dpi=120); fig.patch.set_facecolor("white")  # b5: figure == terminal
ax[0].scatter(x, y, color=CY, s=14); ax[0].plot(x, X @ w_ne, color=V, lw=2)
ax[0].set_title(f"fit:  y = {w_ne[1]:.2f}x + {w_ne[0]:.2f}", color=CY, fontweight="bold")
ax[0].set_xlabel("x"); ax[0].set_ylabel("y")
ax[1].scatter(g0[:, 0], g0[:, 1], color=CY, s=14, label="class 0"); ax[1].scatter(g1[:, 0], g1[:, 1], color=PK, s=14, label="class 1")
xx = np.array([-0.5, 4.5]); ax[1].plot(xx, -(wl[0] + wl[1] * xx) / wl[2], color=V, lw=2)
ax[1].set_title(f"logistic boundary   acc = {acc:.2f}", color=CY, fontweight="bold")
ax[1].set_xlabel("feature 1"); ax[1].set_ylabel("feature 2"); ax[1].legend(loc="upper left", fontsize=8)
fig.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, dpi=120, facecolor="white")
print(f"saved {FIG.name}  (slope {w_ne[1]:.3f}, intercept {w_ne[0]:.3f}, acc {acc:.3f})")


In [ ]:
%%writefile ep28_svm_trees.py
"""
M28 · The Math of SVMs, Trees & Ensembles — three classical workhorses, three small pieces of math.
SVM: maximize the margin (widest gap to the nearest points); the hinge loss max(0, 1 - y(w.x+b))
is the bill for any point that crowds or crosses the margin. Tree: score a split by information
gain = parent impurity minus weighted child impurity, where impurity is entropy (bits). Ensembles:
bagging averages many models to cut variance; boosting chains them to cut bias. Seeded, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m28.png"

rng = np.random.default_rng(28)                              # b1: seed (points fixed below)
X = np.array([[2, 1], [3, 2], [1, 1], [6, 5], [7, 7], [5, 6]], float)  # b1: 6 points, 2 classes
y = np.array([-1, -1, -1, 1, 1, 1])                          # b1: labels +-1
void = rng                                                  # seed documented; points are fixed

w = np.array([1.0, 1.0]); b = -5.5                          # b2: a candidate hyperplane
margins = y * (X @ w + b)                                    # b2: functional margins y(w.x+b)
hinge = np.maximum(0, 1 - margins)                          # b3: hinge loss per point

def entropy(labels):                                        # b4: tree impurity (bits)
    p = np.mean(labels == 1)
    if p in (0.0, 1.0):
        return 0.0
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

parent = entropy(y)                                         # b4: impurity before split
left = y[X[:, 0] <= 4]; right = y[X[:, 0] > 4]              # b4: split feature x1 at 4
child = (len(left) / 6) * entropy(left) + (len(right) / 6) * entropy(right)
ig = parent - child                                        # b5: information gain

print(f"hinge loss total = {hinge.sum():.4f}   (margin violations: {(hinge > 0).sum()})")
print(f"parent entropy = {parent:.4f} bits   child entropy = {child:.4f} bits")
print(f"information gain = {ig:.4f} bits")

CY, GD = "#34D6FF", "#ffd166"
plt.figure(figsize=(6, 4), dpi=120); plt.gcf().patch.set_facecolor("white")  # b5: figure==terminal
for cls, col, mk in [(-1, CY, "o"), (1, GD, "s")]:
    plt.scatter(X[y == cls, 0], X[y == cls, 1], c=col, s=90, marker=mk, edgecolors="#222", label=f"class {cls}")
xx = np.linspace(0, 8, 50); plt.plot(xx, (-b - w[0] * xx) / w[1], "--", c=CY, lw=2, label="boundary")
plt.title(f"SVM boundary   |   info gain = {ig:.2f} bits", fontweight="bold")
plt.xlabel("x1"); plt.ylabel("x2"); plt.xlim(0, 8); plt.ylim(0, 8); plt.legend(loc="upper left", fontsize=8)
plt.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG, dpi=120, facecolor="white")
print(f"saved {FIG.name}  (hinge {hinge.sum():.4f}, violations {(hinge > 0).sum()}, IG {ig:.4f})")


In [ ]:
%%writefile ep29_neural_cnn.py
"""
M29 · The Math of Neural Networks & CNNs — a network is function composition.
Each layer = linear step (multiply by a weight matrix, add a bias) THEN a nonlinearity (ReLU,
which zeros negatives). The bend is non-negotiable: without it, stacked linear layers collapse
algebraically into a single matrix. Convolution is the same multiply-and-add, but one small kernel
is slid across the input (weight sharing) — a "valid" conv's output length is input - kernel + 1.
Runs a tiny 2-layer MLP forward by hand, then convolves a 1D signal with an edge kernel. Seeded.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m29.png"

rng = np.random.default_rng(29)                          # b1: seed (weights fixed below)
relu = lambda z: np.maximum(0, z)                        # b1: the nonlinearity
void = rng

x  = np.array([1.0, -2.0])                               # b2: a 2-feature input
W1 = np.array([[1.0, 1.0], [1.0, -1.0]]); b1 = np.array([0.0, 1.0])   # b2: layer 1
W2 = np.array([2.0, -1.0]); b2 = 0.5                     # b2: layer 2 (to scalar)

h_pre = W1 @ x + b1                                      # b3: linear part
h     = relu(h_pre)                                      # b3: nonlinearity
out   = W2 @ h + b2                                      # b3: final linear -> scalar

signal = np.array([0, 0, 1, 1, 1, 0, 0, 1, 1, 0], float)  # b4: a 1D signal (edges)
kernel = np.array([-1, 1], float)                          # b4: edge detector
conv   = np.array([kernel @ signal[i:i + 2] for i in range(len(signal) - 1)])  # b4: valid conv

print(f"layer1 pre = {h_pre}   after relu = {h}")
print(f"network output = {out:.4f}")
print(f"conv (edge map) = {conv.astype(int)}")
print(f"output length {len(conv)} = input {len(signal)} - kernel {len(kernel)} + 1")

CY, GD = "#34D6FF", "#ffd166"
fig, ax = plt.subplots(1, 2, figsize=(7, 3), dpi=120); fig.patch.set_facecolor("white")  # b5: figure==terminal
ax[0].bar(range(len(signal)), signal, color=CY); ax[0].set_title("input signal", fontweight="bold")
ax[0].set_xlabel("position")
ax[1].bar(range(len(conv)), conv, color=GD); ax[1].set_title(f"conv (edge map), len = {len(conv)}", fontweight="bold")
ax[1].set_xlabel("position"); ax[1].axhline(0, color="#888", lw=0.8)
plt.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG, dpi=120, facecolor="white")
print(f"saved {FIG.name}  (out {out:.4f}, conv len {len(conv)})")


In [ ]:
%%writefile ep30_embeddings.py
"""
M30 · Embeddings & the Geometry of Meaning — turn a word/sentence/image into a vector arranged so
similar things land near each other. "Near" = small angle, measured by cosine similarity (dot product
over the product of lengths): 1 for aligned, 0 for perpendicular, -1 for opposite — direction, not
magnitude. Because directions encode attributes, you can do arithmetic on meaning: king - man + woman
lands on queen. Same geometry powers nearest-neighbor retrieval (vector search / RAG). Seeded, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m30.png"

rng = np.random.default_rng(30)                          # b1: seed (embeddings fixed below)
void = rng
# b1: toy 2D embeddings on axes [royalty, gender]
emb = {"king": [1.0, 1.0], "queen": [1.0, -1.0],
       "man": [0.0, 1.0], "woman": [0.0, -1.0], "throne": [0.9, 0.05]}
V = {k: np.array(v) for k, v in emb.items()}

def cosine(a, b):                                        # b2: cosine similarity
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

cs_kq = cosine(V["king"], V["queen"])                    # b3: king vs queen
cs_kt = cosine(V["king"], V["throne"])                   # b3: king vs throne

analogy = V["king"] - V["man"] + V["woman"]              # b4: king - man + woman
cands = {k: cosine(analogy, v) for k, v in V.items() if k not in ("king", "man", "woman")}  # b4
best = max(cands, key=cands.get)                         # b5

print(f"cos(king, queen)  = {cs_kq:.4f}")
print(f"cos(king, throne) = {cs_kt:.4f}")
print(f"king - man + woman = {analogy}")
print(f"nearest word = '{best}'  (cos = {cands[best]:.4f})")

CY, GD = "#34D6FF", "#ffd166"
plt.figure(figsize=(5, 5), dpi=120); plt.gcf().patch.set_facecolor("white")  # b5: figure==terminal
for k, v in V.items():
    plt.arrow(0, 0, v[0], v[1], head_width=0.05, color=CY, length_includes_head=True)
    plt.text(v[0] * 1.12, v[1] * 1.12, k, color=CY, fontsize=11, ha="center")
plt.scatter(*analogy, c=GD, s=140, zorder=5, edgecolors="#222", label="king - man + woman")
plt.axhline(0, c="#999", lw=0.6); plt.axvline(0, c="#999", lw=0.6)
plt.title(f"analogy lands on '{best}'", fontweight="bold")
plt.xlabel("royalty"); plt.ylabel("gender"); plt.xlim(-0.4, 1.5); plt.ylim(-1.5, 1.5); plt.legend(loc="lower left", fontsize=8)
plt.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG, dpi=120, facecolor="white")
print(f"saved {FIG.name}  (cos kq {cs_kq:.4f}, kt {cs_kt:.4f}, nearest {best} {cands[best]:.4f})")


In [ ]:
%%writefile ep31_fourier_graph_markov.py
"""
M31 · Fourier Transform, Graph Theory & Markov Chains — three pieces of structure, one shared engine
(linear algebra applied to structure). Fourier: a signal is a sum of pure frequencies; the DFT reveals
which ones (the trick behind transformer positional encodings). Graph: relationships stored as an
adjacency matrix (raw material for graph neural nets). Markov chain: a transition matrix where the next
step depends only on now; power-iterate it and it forgets its start, settling to a stationary
distribution (exactly PageRank). Seeded, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m31.png"

rng = np.random.default_rng(31)
void = rng
P = np.array([[0.8, 0.15, 0.05],
              [0.2, 0.7, 0.1],
              [0.1, 0.3, 0.6]])            # b1: 3-state transition matrix (rows sum to 1)
assert np.allclose(P.sum(axis=1), 1.0)

p = np.array([1.0, 0.0, 0.0])                # b2: start fully in state 0
for _ in range(100):                         # b3: power-iterate to stationary dist
    p = p @ P
pi = p / p.sum()

t = np.arange(64)                            # b4: build a signal: 3 Hz + soft 7 Hz
sig = np.sin(2 * np.pi * 3 * t / 64) + 0.5 * np.sin(2 * np.pi * 7 * t / 64)
mag = np.abs(np.fft.rfft(sig))              # b4: DFT magnitudes
peak = int(np.argmax(mag))                  # b5: dominant frequency bin

print("stationary pi =", np.round(pi, 4))
print("pi row-check  =", round(float(pi.sum()), 4))
print("DFT peak bin  =", peak, "  (Hz =", peak, ")")

PK = "#FF6EC7"
fig, ax = plt.subplots(1, 2, figsize=(8, 3.2), dpi=120); fig.patch.set_facecolor("white")
ax[0].bar(range(3), pi, color=PK); ax[0].set_title("Markov stationary pi", fontweight="bold")
ax[0].set_xticks(range(3)); ax[0].set_xticklabels(["s0", "s1", "s2"]); ax[0].set_ylim(0, 1)
for i, v in enumerate(pi):
    ax[0].text(i, v + 0.03, f"{v:.3f}", ha="center", fontsize=9)
ax[1].stem(mag, linefmt=PK, markerfmt="o", basefmt=" "); ax[1].set_title("DFT magnitude (peak at bin 3)", fontweight="bold")
ax[1].set_xlabel("freq bin")
plt.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG, dpi=120, facecolor="white")
print(f"saved {FIG.name}  (pi {np.round(pi, 4)}, peak bin {peak})")


In [ ]:
%%writefile ep32_attention.py
"""
M32 · Attention & Transformer Mathematics — how a transformer lets every token look at every other.
Each token emits a query (what am I looking for), a key (what do I offer), a value (what I'll hand over).
Compare each query to all keys via dot product, scale by sqrt(d) (keeps scores from exploding as d grows),
softmax into weights that sum to 1, and take a weighted average of the values. Scaled dot-product
attention = softmax(Q K^T / sqrt(d)) V — a fully differentiable, learned soft lookup. Seeded, offline.
Runnable: pip install torch matplotlib
"""
import pathlib
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m32.png"

torch.manual_seed(32)
n, d = 4, 8                                   # b1: 4 tokens, model dim 8
X = torch.randn(n, d)                         # b1: token embeddings
Wq, Wk, Wv = (torch.randn(d, d) for _ in range(3))  # b2: projection matrices

Q, K, V = X @ Wq, X @ Wk, X @ Wv             # b3: queries, keys, values
scores = Q @ K.T / (d ** 0.5)                # b3: scaled dot-product QK^T/sqrt(d)
A = F.softmax(scores, dim=-1)                # b4: attention weights (rows sum to 1)
out = A @ V                                   # b4: weighted sum of values

row0 = [round(float(x), 3) for x in A[0]]
print("attention rows sum =", [round(float(s), 3) for s in A.sum(-1)])
print("token0 attends to  =", row0)
print("argmax per query    =", A.argmax(-1).tolist())
print("output shape        =", tuple(out.shape))

plt.figure(figsize=(4.2, 3.6), dpi=120); plt.gcf().patch.set_facecolor("white")
plt.imshow(A.detach(), cmap="magma"); plt.colorbar()
plt.title("attention  softmax(QK^T / sqrt(d))", fontweight="bold")
plt.xlabel("key token"); plt.ylabel("query token")
plt.xticks(range(n)); plt.yticks(range(n))
plt.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG, dpi=120, facecolor="white")
print(f"saved {FIG.name}  (token0 row {row0}, argmax {A.argmax(-1).tolist()})")


In [ ]:
%%writefile ep33_diffusion.py
"""
M33 · Diffusion Model Mathematics — a model learns to create by first learning to destroy.
Forward process: add a little Gaussian noise each step until the signal is gone; a closed-form
shortcut jumps to any noise level at once via alpha-bar (cumulative signal kept). Reverse process:
a network looks at a noisy image and predicts the noise inside it; subtract it step by step and a
picture emerges from static. Training objective is embarrassingly simple — predict the noise,
minimize squared error ||eps - eps_hat||^2. This powers Stable Diffusion. Seeded, offline.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m33.png"

rng = np.random.default_rng(33)
x0 = np.array([1.0, -1.0])                    # b1: a tiny clean "image" (2 pixels)
T = 50                                         # b1: number of noising steps
betas = np.linspace(1e-4, 0.2, T)            # b2: variance schedule
alpha_bar = np.cumprod(1.0 - betas)          # b2: cumulative signal retained

t = 30                                         # b3: pick a timestep
eps = rng.standard_normal(x0.shape)          # b3: the true noise we add
xt = np.sqrt(alpha_bar[t]) * x0 + np.sqrt(1 - alpha_bar[t]) * eps  # b3: closed-form noising

eps_hat = (xt - np.sqrt(alpha_bar[t]) * x0) / np.sqrt(1 - alpha_bar[t])  # b4: recover noise
loss = np.mean((eps - eps_hat) ** 2)         # b4: objective ||eps - eps_hat||^2

print("alpha_bar[0]  =", round(float(alpha_bar[0]), 4))
print("alpha_bar[t]  =", round(float(alpha_bar[t]), 4))
print("alpha_bar[-1] =", round(float(alpha_bar[-1]), 4))
print("recover loss  =", round(float(loss), 8))

PK = "#FF6EC7"
plt.figure(figsize=(6, 3.6), dpi=120); plt.gcf().patch.set_facecolor("white")
plt.plot(alpha_bar, color=PK, lw=2, label="alpha_bar (signal kept)")
plt.axvline(t, color="#7d8693", ls="--"); plt.scatter([t], [alpha_bar[t]], color=PK, zorder=3)
plt.annotate(f"t={t}: {alpha_bar[t]:.4f}", (t, alpha_bar[t]), textcoords="offset points", xytext=(10, 10), color=PK, fontsize=9)
plt.title("forward diffusion: signal decays to noise", fontweight="bold")
plt.xlabel("timestep t"); plt.ylabel("alpha_bar_t"); plt.ylim(0, 1); plt.legend()
plt.tight_layout()
FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG, dpi=120, facecolor="white")
print(f"saved {FIG.name}  (ab0 {alpha_bar[0]:.4f}, ab30 {alpha_bar[t]:.4f}, ab-1 {alpha_bar[-1]:.4f}, loss {loss:.2e})")


In [ ]:
%%writefile ep34_rl.py
"""
M34 · Reinforcement Learning Math + Course Conclusion — math for ACTING, not just predicting.
Cast the world as a Markov decision process: in a state you take an action, collect a reward, land
in a new state. The return is the discounted sum of future rewards; the value function is the
expected return from a state. Bellman: V(s) = R + gamma * V(s') — today's value is the immediate
reward plus the discounted value of where you land, a recursion solved by value iteration. Policy
gradients push the policy toward higher-return actions — exactly the machinery behind RLHF. Seeded.
Runnable: pip install numpy matplotlib
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).resolve().parent
FIG = pathlib.Path(".") / "m34.png"

# b1: a 3-state chain S0 -> S1 -> S2(goal). reward +1 for reaching S2.
gamma = 0.9
R = np.array([0.0, 0.0, 1.0])     # reward at each state
nxt = np.array([1, 2, 2])         # deterministic "go right"; S2 absorbs

# b2: value iteration = repeated Bellman backup V(s) = R(s') + gamma*V(s')
V = np.zeros(3); hist = [V.copy()]
for _ in range(8):
    Vn = V.copy()
    for s in range(2):            # S2 terminal, value stays 0 baseline
        Vn[s] = R[nxt[s]] + gamma * V[nxt[s]]
    V = Vn; hist.append(V.copy())
hist = np.array(hist)

# b3: print converged values
print(f"V(S0) = {V[0]:.4f}")
print(f"V(S1) = {V[1]:.4f}")
print(f"sweeps run = {len(hist) - 1}")

PK, PU = "#FF6EC7", "#bd7dff"
fig, ax = plt.subplots(figsize=(6, 4), dpi=120); fig.patch.set_facecolor("white"); ax.set_facecolor("white")
for s in range(2):
    ax.plot(hist[:, s], marker="o", color=PK if s == 1 else PU, lw=2, label=f"V(S{s})")
ax.set_title(f"Value iteration: V(S0)={V[0]:.2f}, V(S1)={V[1]:.2f}", fontweight="bold")
ax.set_xlabel("sweep"); ax.set_ylabel("value"); ax.set_ylim(-0.05, 1.1); ax.legend(loc="lower right")
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, dpi=120, bbox_inches="tight", facecolor="white")
print(f"saved {FIG.name}  (V(S0)={V[0]:.4f}, V(S1)={V[1]:.4f})")


## M01 · Course Intro + Numbers & Algebra for ML

Welcome to Mathematics for Machine Learning. The promise is simple: we start from the most primitive idea -- a number -- and climb, one honest step at a time, to the math that powers modern AI. Six stages: Foundations, Linear Algebra, Calculus, Probability, Optimization, and the ML math that ties it together. The discipline is intuition first, then the real math, then code you can run. And the punchline of this very first episode is the thing people find hardest to believe: a neural network is not magic. It is billions of numbers being multiplied and added. A weight is a number. A prediction is a sum. A loss is one number measuring how wrong you were. Learn to read numbers and algebra fluently, and you have already learned the alphabet of every model.

In [ ]:
!python ep01_neuron.py

loop sum   = -2.0000
dot sum    = -2.0000
terms      = [ 1.  -1.5 -1.5]
loss (MSE) = 9.0000
log loss   = 2.1972
wrote m01.png  (sum=-2.0, loss=9.0)


## M02 · Functions & Graphs — the Shape of Models

Last episode a neuron was a weighted sum -- a single number. This episode we give that number a life: we feed it through a function and watch it become a shape. A function is just a rule: hand it an input, it hands back one output. Linear functions draw straight lines and can only bend the world so far; the moment you want curves, decisions, or saturation, you need nonlinear functions -- and the two superstars of ML, sigmoid and ReLU, are exactly that. The deepest idea here is composition: chain simple functions and you can approximate almost anything. That is literally what a neural network is -- a function f(x; w) built by stacking functions. A model is not a mysterious entity; it is a graph you could sketch by hand.

In [ ]:
!python ep02_functions.py

linear(2)  = 3.0000
sigmoid(2) = 0.8808
relu(2)    = 2.0000
sigmoid(linear(2)) nested = 0.9526
sigmoid(linear(2)) byhand = 0.9526
saved m02.png  (sigmoid(linear(2)) = 0.953)


## M03 · Trigonometry & Geometry for ML

We've drawn functions as shapes; now we step into the space those shapes live in. Geometry gives ML two superpowers. The first is distance -- how far apart two points are -- which is how a model knows "near" from "far." The second is angle -- how aligned two directions are -- which the dot product hands us for free. Aligned vectors point the same way; perpendicular ones share nothing. That single idea, the cosine of the angle between two vectors, is cosine similarity, the workhorse of every embedding system: words, images, recommendations all compared by the angle between them. And a straight line or flat plane in this space is a decision boundary -- one side is class A, the other class B. Geometry is the language a model uses to compare.

In [ ]:
!python ep03_geometry.py

distance(a,c) = 4.4721
cos(a,b) = 0.9899  angle = 8.13 deg
cos(a,c) = 0.0000  angle = 90.00 deg
saved m03.png  (cos(a,b)=0.99, cos(a,c)=0.00)


## M04 · Set Theory & Mathematical Logic

Almost everything in ML is a collection of things and a rule for sorting them. A dataset is a set; a train/test split partitions that set; a model's predictions form a subset of "things called positive." Set theory gives us the grammar for talking about membership and overlap, and logic gives us the grammar for the rules — and once you see precision and recall as the overlap between "actually positive" and "predicted positive," the whole evaluation story becomes one Venn diagram.

In [ ]:
!python ep04_sets.py

actual=7 predicted=9 TP=3
precision=0.333
recall=0.429
saved m04.png  (precision=0.333, recall=0.429)


## M05 · Sequences, Series & Limits

A sequence is just numbers in order; a series is what you get when you keep adding them up. The deep question is what happens at the end of forever — does the running total settle on a value, or run off to infinity? That settling is a *limit*, and it's the single most important idea in all of calculus: the limit of a shrinking step is what becomes the derivative that powers gradient descent, and a geometric series with ratio less than one is exactly the discounted sum of future rewards in RL.

In [ ]:
!python ep05_series.py

limit a/(1-r) = 1.000000
partial_sum[11] = 0.999756
gap = 0.000244
saved m05.png  (limit=1.000, final=0.9998)


## M06 · Vectors & Vector Spaces

A vector is a single object that carries several numbers at once — a point in space, an arrow from the origin. Two operations define everything: you can add vectors tip-to-tail, and you can scale them longer or shorter. The set of all places you can reach by combining a few vectors is their *span*, and the smallest independent set that spans a space is its *basis*. This is the native language of ML: every data point, every word embedding, every layer's activation is a vector, and "feature space" is the vector space they all live in.

In [ ]:
!python ep06_vectors.py

u+v = [4. 6.]
2u  = [6. 4.]
det = 10.0  independent = True
saved m06.png  (u+v=[4. 6.], det=10.0)


## M07 · Matrices & Matrix Operations

A matrix is just a rectangle of numbers, but it earns its keep as the universal verb of machine learning: multiply by it and you transform a whole batch of inputs at once. Every linear layer in every neural net is literally Wx+b, a matrix times your data plus a bias. Matmul is the single operation a GPU runs billions of times a second, so understanding it is understanding the engine room of deep learning.

In [ ]:
!python ep07_matrices.py

X shape: (4, 3)  W shape: (3, 3)  Y shape: (4, 3)
rank(W): 3
max reconstruction error: 0.0
saved m07.png  (Y shape (4, 3), rank 3, err 4.440892098500626e-16)


## M08 · Linear Transformations

A matrix isn't just a grid of numbers, it's an instruction for warping space: every point gets sent somewhere new, but straight lines stay straight and the origin stays put. Rotation, scaling, and shear are all matrices, and applying them in sequence is just multiplying them together. The determinant tells you how much area got stretched or squished. This is exactly why a deep stack of linear layers with no nonlinearity is secretly one layer: the product of all those matrices is itself a single matrix.

In [ ]:
!python ep08_transforms.py

det(R): 1.0
det(S): 3.0
det(M): 3.0
det(R)*det(S): 3.0
match: True
saved m08.png  (det(M)=3.00 = 1 x 3)


## M09 · Norms, Dot Products & Projections

Once vectors live in space, you want to measure them: how long is this vector, and how aligned are these two? The L2 norm is plain length; the L1 norm sums absolute parts and quietly drives sparsity. The dot product is the swiss-army knife: it gives length when a vector meets itself, gives the cosine of the angle when normalized, and reads zero when two vectors are perpendicular. Projection answers "how much of A points along B," which is the literal math behind L2 regularization, cosine similarity, PCA components, and attention scores.

In [ ]:
!python ep09_norms.py

L1(A): 7.0
L2(A): 5.0
A . B: 12.0
cosine: 0.6
proj of A onto B: [3. 0.]
|proj|: 3.0
saved m09.png  (cos=0.60, |proj|=3.0)


## M10 · Eigenvalues & Eigenvectors

Most vectors get knocked sideways when a matrix acts on them. A precious few keep pointing the exact same way and just get longer or shorter. Those invariant directions are the eigenvectors, and the amount of stretch is the eigenvalue. They are the secret skeleton of a linear map: once you know how a matrix behaves along these special directions, you know how it behaves everywhere. The ML payoff is huge. The eigenvectors of a covariance matrix are the principal axes of your data, which is exactly what PCA finds, and the whole family of spectral methods rides on the same idea.

In [ ]:
!python ep10_eigen.py

eigenvalues: [3. 1.]
top eigenvector v: [0.7071 0.7071]
A v             : [2.1213 2.1213]
lambda * v      : [2.1213 2.1213]
residual ||Av - lambda v|| = 0.0
saved m10.png  (lambda1=3.0, residual=0.0e+00)


## M11 · Singular Value Decomposition (SVD)

Eigenvectors only work nicely for square, symmetric matrices. SVD says: every matrix, any shape, factors into three honest pieces, a rotation U, a pure stretch S along clean axes, and another rotation V^T. The stretch amounts, the singular values, are sorted biggest to smallest, and they tell you exactly how much information lives in each direction. Keep the top few and you reconstruct most of the matrix from a sliver of numbers. That single trick powers dimensionality reduction, latent semantic analysis, image compression, and the low-rank intuition behind LoRA fine-tuning.

In [ ]:
!python ep11_svd.py

singular values: [7.803 0.128 0.093 0.014]
rank-2 relative error: 0.012
energy captured (k=2): 0.9999
saved m11.png  (s1=7.80, energy=0.9999, err=0.0120)


## M12 · Principal Component Analysis (PCA)

Real data is usually squashed into a thin slab inside a high-dimensional box; most directions carry almost nothing. PCA finds the few directions where the data actually spreads. You center the data, compute its covariance matrix, take its eigenvectors, those are the principal components, and the eigenvalues tell you how much variance each one explains. Project onto the top k and you have compressed the data while keeping its shape. That is how we squeeze features, plot 100-dimensional embeddings in 2D, and even denoise by dropping the tiny components.

In [ ]:
!python ep12_pca.py

eigenvalues: [7.586 1.094]
variance explained: [0.874 0.126]
PC1 direction: [0.867 0.498]
saved m12.png  (PC1 87.4%, PC1 dir [0.867 0.498])


## M13 · Tensors & Broadcasting

A tensor is just an n-dimensional array, and its *shape* is the contract that tells you what each axis means: which one is the batch, which is the channel, which are the pixels. Broadcasting is the rule that lets a small array act as if it were stretched to match a big one, so a single bias vector can be added to a whole batch of activations without ever writing a loop. The ML payoff: every layer in a neural net is shape-juggling — get shapes right and the math is forced to be right too.

In [ ]:
!python ep13_tensors.py

X.shape    = (4, 3)
bias.shape = (3,)
Y.shape    = (4, 3)
X row0     = [7.8 7.7 7.3]
Y row0     = [ 8.8  9.7 10.3]
reshape ok = True
same bias all rows = True
saved m13.png  (Y-X every row = bias, same_all=True)


## M14 · Derivatives & Limits

A derivative is just the slope of a curve at a single point — the rate at which the output changes for a tiny nudge in the input. We get it by taking a secant line through two nearby points and sliding them together until the gap vanishes; the limit of that slope is the derivative. The ML payoff: training is downhill walking on a loss curve, and the derivative is the compass that says which way is down and how steep — including the sigmoid derivative that flows through every backprop step.

In [ ]:
!python ep14_derivatives.py

x0           = 1.5
numerical f' = 3.00001
exact f'     = 3.0
abs error    = 1.00e-05
saved m14.png  (numerical 3.00001 vs exact 3.0, err 1.0e-05)


## M15 · Multivariable Calculus & Gradients

When a function has many inputs, a single slope isn't enough — each input has its own slope, found by nudging that one variable and freezing the rest. That's a partial derivative. Collect them into a vector and you get the gradient, which points in the direction of steepest ascent; its negative is the direction of steepest descent. The ML payoff: a loss depends on millions of weights, and the gradient is the one vector that tells gradient descent how to move every weight at once to go downhill fastest.

In [ ]:
!python ep15_gradients.py

point        = (1.0, 2.0)
exact grad   = (2.0, 4.0)
numeric grad = (2.000, 4.000)
|grad|       = 4.4721
saved m15.png  (grad (2,4) outward, perpendicular to rings, |grad|=4.4721)


## M16 · The Chain Rule

Most useful functions are nested: a layer feeds a layer feeds a loss. The chain rule says the derivative of a nest is just the product of the derivatives at each link. Stack four functions, get four numbers, multiply them. That single multiplicative rule, applied link by link from the loss back to every weight, is the literal mathematical heart of backpropagation. Learn this and you understand how gradients flow.

In [ ]:
!python ep16_chainrule.py

g(x0)      = 7.0000
chain dy/dx= 42.0000
numeric    = 42.0000
match       = True
saved m16.png  (chain == numeric, dy/dx@2 = 42.0)


## M17 · Computational Graphs & Automatic Differentiation

The chain rule by hand gets unwieldy fast. So we draw the computation as a graph: each operation is a node, each input an edge. Forward pass fills in values. Then reverse-mode autodiff walks the graph backward once, multiplying local gradients along the way, and hands you the derivative with respect to every input in a single sweep. That single backward sweep is exactly what PyTorch does when you call .backward() — and it is why training models with millions of parameters is even feasible.

In [ ]:
!python ep17_autodiff.py

z              = 3.9975
dz/dx autograd = 2.0707   hand = 2.0707
dz/dy autograd = 1.5000   hand = 1.5000
match           = True
saved m17.png  (autograd == hand: dz/dx=2.0707, dz/dy=1.5000)


## M18 · Backpropagation Mathematics

Backprop is the chain rule wearing a neural-net costume. Forward, a small MLP turns inputs into a prediction and a loss. Backward, you start from the loss gradient and push it through each layer: at every step the local gradient (an activation derivative or a weight transpose) multiplies the signal from above, and the weight gradient is simply that upstream signal times the layer's input. Repeated multiplication explains everything — including why gradients can shrink to nothing or blow up as nets get deep.

In [ ]:
!python ep18_backprop.py

loss        = 0.0568
gW2 hand    = [-0.1434 -0.0842]
gW2 autograd= [-0.1434 -0.0842]
gW1 hand    = [-0.0412  0.0824  0.0379 -0.0758]
gW1 autograd= [-0.0412  0.0824  0.0379 -0.0758]
match        = True
saved m18.png  (hand == autograd: True)


## M19 · Gradient Descent & its Variants

Every model you have ever heard of is trained the same way: start somewhere, look at which direction the loss falls fastest, take a small step that way, repeat. That direction is the negative gradient; the step size is the learning rate. The variants — stochastic, mini-batch, momentum, Adam — are all answers to one practical question: how do we take that step cheaply, smoothly, and at the right size on a messy, high-dimensional loss surface like a neural network's?

In [ ]:
!python ep19_gradient_descent.py

start x = 5.0000  loss = 25.0000
step  1 x = 4.0000  loss = 16.0000
step  5 x = 1.6384  loss = 2.6844
final x = 0.0007  loss = 0.000000
saved m19.png  (start loss 25.0 -> final 0.000000)


## M20 · Convex Optimization & Numerical Methods

Some loss surfaces are a single clean bowl — convex. From anywhere you walk downhill, you reach the one true bottom; logistic and linear regression live here, so their training has a guaranteed answer. Neural networks are nonconvex: a rugged landscape of many valleys, where downhill only promises a local minimum, not the best one. Knowing which world you're in tells you whether your optimizer's answer is the answer — or just a good-enough resting place.

In [ ]:
!python ep20_convexity.py

start -6.0 -> x = -1.306  loss = -7.946
start  4.0 -> x = 3.837  loss = 8.316
same algorithm, different minima: 16.261 apart
saved m20.png  (two starts -> losses -7.946 vs 8.316)


## M21 · Probability & Random Variables

Probability is the math of uncertainty. List everything that could happen (the sample space), assign each a weight between 0 and 1 that sums to one, and you have a model of chance. A random variable turns those outcomes into numbers; its expectation is the long-run average you'd see, and its variance measures how wildly it swings. This is the language every classifier speaks — its output is literally a probability over classes, and its training minimizes the expected loss.

In [ ]:
!python ep21_probability.py

theory   E[X] = 4.7000   Var[X] = 2.1100
sampled  mean = 4.7072   var    = 2.1008
agree to ~2 decimals: 0.007, 0.009
saved m21.png  (E[X]=4.7000, Var=2.1100; sampled 4.7072/2.1008)


## M22 · Probability Distributions

A probability distribution is just an honest budget of belief: it spreads a total of 1.0 across the outcomes that could happen. For things you can count -- a coin, a class label -- the budget sits on points (a PMF). For things you can measure -- a height, a noise value -- it smears into a smooth curve where area, not height, is the probability (a PDF). The Normal earns its fame because sums of many small effects pile up into its bell, which is exactly why we model noise as Gaussian, read a softmax as a categorical distribution, and write logistic regression's loss as a Bernoulli likelihood.

In [ ]:
!python ep22_distributions.py

Bernoulli p_hat = 0.2999  (true 0.3)
Gaussian  mean  = 0.0179  std = 1.0010
Softmax   probs = [0.659  0.2424 0.0986]  sum = 1.0000
saved m22.png  (p_hat=0.2999, gauss 0.0179/1.0010, cat [0.659  0.2424 0.0986])


## M23 · Bayes' Theorem

Bayes' theorem is the math of changing your mind. You start with a prior belief, you see some evidence, and the *likelihood* of that evidence under each hypothesis pulls your belief toward whichever hypothesis explains it best -- giving a posterior. The headline lesson is base rates: when a condition is rare, even an accurate test produces mostly false alarms. In ML this is the entire Naive Bayes classifier, the spirit of Bayesian updating, and the intuition behind a posterior over a model's weights.

In [ ]:
!python ep23_bayes.py

Prior  P(disease)      = 0.0100
Evidence P(positive)   = 0.0594
Posterior P(disease|+) = 0.1667
saved m23.png  (P(+)=0.0594, posterior=0.1667; true+ 0.0099 vs false+ 0.0495)


## M24 · Statistics, Sampling & Estimation

You almost never see the whole population -- you see a sample, and you guess the truth from it. Two laws make that guessing trustworthy. The Law of Large Numbers says a sample mean converges to the true mean as you collect more data. The Central Limit Theorem says the *spread* of sample means shrinks like one-over-root-n and goes bell-shaped, no matter the original distribution. That is exactly why mini-batch gradients are noisy but unbiased estimates of the full gradient, why bigger batches are steadier, and why bagging resamples to trade variance for stability.

In [ ]:
!python ep24_statistics.py

Running mean (2000 draws) = 5.0535  (true 5.0)
Mean of sample-means      = 4.9960
Std of sample-means       = 0.3699  (theory 0.3651)
saved m24.png  (running 5.0535, means 4.9960/0.3699 vs theory 0.3651)


## M25 · Hypothesis Testing

We almost always have *some* difference in the numbers: model A scored 84.1%, model B scored 85.3%. The question is never "is there a gap?" — there always is. The question is "would noise this big show up by accident, even if the two were truly identical?" Hypothesis testing answers that. We assume the boring story (the null: no real difference), compute how surprising our data would be under it (the p-value), and reject the boring story only when the data is too surprising to ignore. The ML payoff: this is exactly the math behind trusting an eval delta or an A/B test instead of chasing noise.

In [ ]:
!python ep25_hypothesis.py

mean A = 0.8096  mean B = 0.8478  gap = 0.0382
t = 3.880  df = 98  p = 0.0002
95% CI for gap = [0.0187, 0.0577]
verdict: B is significantly better
saved m25.png  (gap 0.0382, t 3.880, p 0.0002)


## M26 · Information Theory — Entropy, Cross-Entropy, KL

Information is surprise. A coin that always lands heads tells you nothing; a fair coin tells you one bit each flip. Entropy is the average surprise of a distribution — its irreducible uncertainty. Cross-entropy measures what you pay when you encode reality with the *wrong* beliefs: the truer your model, the cheaper. KL divergence is exactly that overpayment — cross-entropy minus entropy — and it's zero only when your model matches reality. The ML payoff is enormous: cross-entropy is the loss that trains nearly every classifier and every language model, and KL is the regularizer that keeps VAEs, distilled students, and RLHF policies from drifting too far.

In [ ]:
!python ep26_information.py

entropy H(p)        = 1.7427 bits
cross-entropy(p,q)  = 1.7757 bits
KL(p || q)          = 0.0329 bits
check  H + KL        = 1.7757 bits  (== cross-entropy)
identity CE = H + KL holds: True
saved m26.png  (H 1.7427, CE 1.7757, KL 0.0329)


## M27 · The Math of Linear & Logistic Regression

Strip away the hype and almost every model starts here. Linear regression fits a straight line by minimizing squared error — and that has a clean closed-form answer, the normal equations, plus a gradient if you'd rather descend. Logistic regression takes the same linear score w·x+b, squashes it through a sigmoid into a probability, and trains with the cross-entropy loss from last episode. The line where that probability crosses 0.5 is the decision boundary. The ML payoff: these are the two atoms — fit a number, or split into classes — and deep nets are just stacks of this idea with nonlinearities between.

In [ ]:
!python ep27_regression.py

normal eq:  intercept=0.388  slope=1.812
grad desc:  intercept=0.388  slope=1.812
weights match: True
logistic accuracy = 1.000
saved m27.png  (slope 1.812, intercept 0.388, acc 1.000)


## M28 · The Math of SVMs, Trees & Ensembles

Three of classical ML's workhorses rest on three small pieces of math. An SVM doesn't just separate the classes — it pushes the boundary as far from both as possible, and the hinge loss is the bill for any point that violates that margin; kernels let it draw curved boundaries by measuring similarity in a richer space. A decision tree asks: which question cuts the most confusion out of the data? That "confusion" is impurity (gini or entropy), and information gain is the drop. Ensembles then say one tree is shaky, so bagging averages many to kill variance, while boosting chains them to attack bias — and that's why random forests and gradient boosting still win tabular competitions.

In [ ]:
!python ep28_svm_trees.py

hinge loss total = 0.5000   (margin violations: 1)
parent entropy = 1.0000 bits   child entropy = 0.0000 bits
information gain = 1.0000 bits
saved m28.png  (hinge 0.5000, violations 1, IG 1.0000)


## M29 · The Math of Neural Networks & CNNs

A neural network is just function composition: take your input vector, multiply by a weight matrix, add a bias, bend it with a nonlinearity like ReLU, and feed the result into the next layer. Stack a few of these and you can approximate almost any function — but only because of that bend; remove every nonlinearity and the whole tower algebraically flattens into a single matrix. Convolution is the same multiply-and-add, but with one tiny weight pattern slid across the input, reusing the same parameters everywhere. That weight sharing is why a CNN can spot an edge anywhere in an image with a handful of numbers, and pooling shrinks what it sees while keeping the gist.

In [ ]:
!python ep29_neural_cnn.py

layer1 pre = [-1.  4.]   after relu = [0. 4.]
network output = -3.5000
conv (edge map) = [ 0  1  0  0 -1  0  1  0 -1]
output length 9 = input 10 - kernel 2 + 1
saved m29.png  (out -3.5000, conv len 9)


## M30 · Embeddings & the Geometry of Meaning

An embedding turns a word, sentence, or image into a vector — a point in a high-dimensional space arranged so that similar things land near each other. "Near" usually means small angle, measured by cosine similarity: the dot product divided by the lengths, so two vectors pointing the same way score 1 regardless of magnitude. Because directions encode attributes, you can do arithmetic on meaning itself — subtract "man," add "woman," and king slides over to queen. This same geometry is what powers nearest-neighbor retrieval: vector search and RAG embed your query, then return whatever sits closest in the space.

In [ ]:
!python ep30_embeddings.py

cos(king, queen)  = 0.0000
cos(king, throne) = 0.7452
king - man + woman = [ 1. -1.]
nearest word = 'queen'  (cos = 1.0000)
saved m30.png  (cos kq 0.0000, kt 0.7452, nearest queen 1.0000)


## M31 · Fourier Transform, Graph Theory & Markov Chains

Three pieces of structure power modern AI. A Fourier transform takes a signal and tells you which frequencies it is made of — the same trick behind positional encodings in transformers. A graph stores relationships as an adjacency matrix, the raw material for graph neural networks. A Markov chain is a transition matrix where each step depends only on where you are now; multiply it enough times and it forgets where it started, settling into a stationary distribution — exactly the math behind PageRank. Three different objects, one shared engine: linear algebra applied to structure.

In [ ]:
!python ep31_fourier_graph_markov.py

stationary pi = [0.4615 0.3846 0.1538]
pi row-check  = 1.0
DFT peak bin  = 3   (Hz = 3 )
saved m31.png  (pi [0.4615 0.3846 0.1538], peak bin 3)


## M32 · Attention & Transformer Mathematics

Attention is how a transformer lets every word look at every other word and decide what to pay attention to. Each token emits a query (what am I looking for), a key (what do I offer), and a value (what I'll hand over). Compare each query against all keys with a dot product, soften those scores into probabilities with softmax, and use them to take a weighted average of the values. Scaling by the square root of the dimension keeps those scores from exploding as vectors grow. Stack several of these in parallel — multiple heads — and you get the engine inside every GPT: a fully differentiable form of lookup, learned end to end.

In [ ]:
!python ep32_attention.py

attention rows sum = [1.0, 1.0, 1.0, 1.0]
token0 attends to  = [0.0, 0.0, 0.213, 0.787]
argmax per query    = [3, 2, 1, 1]
output shape        = (4, 8)
saved m32.png  (token0 row [0.0, 0.0, 0.213, 0.787], argmax [3, 2, 1, 1])


## M33 · Diffusion Model Mathematics

A diffusion model learns to create by first learning to destroy. The forward process slowly drowns a clean image in Gaussian noise over many steps, until nothing but static remains — and there's a closed-form shortcut to jump to any noise level at once using a "signal-kept" schedule called alpha-bar. The reverse process is where the magic is: a network learns to look at a noisy image and predict the noise inside it. Subtract that predicted noise, step by step, and a picture emerges from static. The training objective is almost embarrassingly simple — predict the noise, minimize the squared error. That single idea powers Stable Diffusion and every modern image generator.

In [ ]:
!python ep33_diffusion.py

alpha_bar[0]  = 0.9999
alpha_bar[t]  = 0.1375
alpha_bar[-1] = 0.0046
recover loss  = 0.0
saved m33.png  (ab0 0.9999, ab30 0.1375, ab-1 0.0046, loss 3.08e-32)


## M34 · Reinforcement Learning Math + Course Conclusion

Reinforcement learning is math for acting, not just predicting. You cast the world as a Markov decision process: in some state you take an action, collect a reward, and land in a new state. The return is the discounted sum of all future rewards, and the value function is the expected return from a state. The Bellman equation says today's value equals the immediate reward plus the discounted value of where you land — a recursion you can solve by iteration. Policy gradients then push the policy toward actions that earned more return. This is exactly the machinery behind RLHF: the reward model scores the LLM's answers, and a policy gradient nudges the model to produce the answers humans prefer.

In [ ]:
!python ep34_rl.py

V(S0) = 0.9000
V(S1) = 1.0000
sweeps run = 8
saved m34.png  (V(S0)=0.9000, V(S1)=1.0000)
